# Auditoria de Carga Horária x Alunos

> **Portfolio pessoal.** Este notebook foi extraído de um projeto real de automação de processos acadêmicos e passou por um processo de anonimização: nomes de instituições, pessoas e caminhos locais foram substituídos por termos genéricos, e todos os outputs de execução (que continham dados reais de alunos/professores) foram removidos. O código e a lógica de negócio foram mantidos como estavam em produção.

Reescrita em Python/pandas de um modelo originalmente feito em Power Query, para auditar carga horária versus alunos em bases com milhões de linhas.

---


# Análise Hierárquica CH x Aluno

Reescrita em Python/pandas do modelo construído no Power Query, para ganho de performance com bases de milhões de linhas.

## O que mudou nesta versão

**1. Correção — o módulo da matrícula passou a ser respeitado.**
Na versão anterior, a contagem de alunos (Etapa 0B) agrupava por `curso × unidade × disciplina`, sem `SEMESTRE`, e a junção com `PRATICAS` (Etapa 0C) usava apenas `COD_CURSO + COD_DISCIPLINA`. Como 43 pares curso-disciplina têm prática em mais de um módulo (37 em dois, 6 em quatro), o contingente inteiro de alunos era replicado em cada um. Efeito medido: das 151.932 matrículas contabilizadas, **16.877 (11%) entravam num módulo diferente daquele em que o aluno está matriculado**. O total correto é 135.055. Agora o `MODULO` faz parte da chave nas duas etapas.

**2. Nova aba `RESUMO_POLO_MODULO`** — uma linha por polo × módulo × curso.
A cidade continua sendo o nível **oficial** do delta: o tutor atende todos os polos do município, então a capacidade é um pool municipal, não uma lotação por unidade física. Esta aba distribui o pool entre os polos proporcionalmente à demanda de cada um — os pesos somam 1, então a soma dos polos reconcilia exatamente com a cidade.

Como o rateio é proporcional à demanda, o *percentual* de cobertura é idêntico em todos os polos de uma cidade para um mesmo curso. A aba mostra **onde o déficit se materializa em horas absolutas** — priorização e logística —, não um diagnóstico independente do que a cidade já revela. O único caso em que um polo acende sozinho é quando há demanda nele e a cidade inteira não tem tutor para aquele curso.

Cada linha traz, além das horas: `TEM_SINERGIA_POLO_CURSO` (SIM se qualquer disciplina do curso naquele polo estiver em sinergia válida), `CAPACIDADE_LPI_MIN`/`MAX`, `QTD_PRATICAS_MODULO_MIN`/`MAX` e `CH_MENTORIA_MODULO_CURSO_POLO`. Os totais do polo são repetidos em cada linha, sufixados `_NAO_SOMAR`.

**3. Correção — CH de preparação ("mentoria") no grão de curso.**
A preparação existia no `detalhe` apenas na forma da linha contabilizadora (`CH_PREPARACAO_MODULO_LINHA`), que carrega a preparação **inteira da oferta** e a atribui a um único curso. A demanda já tinha a versão rateada entre cursos pelo peso de alunos; a preparação, não. Qualquer leitura de mentoria por curso — inclusive em tabela dinâmica sobre `ANALISE_CIDADE_CH` — dava 100% da preparação ao curso dono da linha e **zero ao parceiro de sinergia**. A Etapa 7B cria `CH_MENTORIA_MODULO_AUX_RATEIO_CURSO` pela mesma fórmula da demanda, e o assert 4C guarda a propriedade.

**Nunca some demanda + mentoria**: a CH de preparação está *contida* na CH do tutor, não é um adicional.

**4. Novo modo de projeção do módulo seguinte (desligado por padrão).**
`MODO_ANALISE = 'REALIZADO'` reproduz exatamente o comportamento original. Com `'PROJETADO'`, a demanda passa a ser a estimativa do próximo módulo, calculada pela matriz curricular, e a saída vai para um arquivo separado — os dois nunca se misturam na mesma tabela.

Premissas validadas contra a própria base antes de serem codificadas:

| Verificação | Resultado |
|---|---|
| `MODULO` da matriz (via `CURRICULO`+`DISCIPLINA`) == `SEMESTRE` da matrícula | 99,3% |
| Série cursada avança exatamente +1 por módulo (51→52) | 91,7% |
| Série cursada avança exatamente +1 por módulo (52→53) | 91,6% |

**Atenção ao campo `SERIE_ALUNO_ATUAL`**: é um snapshot e **não** serve para projetar. Coincide com o `SERIE_IDEAL` da disciplina efetivamente cursada em apenas 41,2% das linhas, e entre os módulos 52 e 53 permanece inalterado em 96% dos casos. A série é derivada da matriz (moda do `SERIE_IDEAL` das disciplinas do aluno no módulo).

**5. Célula final de validação** com seis conferências, quatro delas com `assert` que interrompem a execução:
1. Demanda x capacidade por módulo, com cobertura
2. Déficit **real** — soma apenas dos negativos, porque sobra numa cidade não cobre falta em outra; o delta agregado esconde o problema
3. Nenhum tutor com ociosidade maior que a própria capacidade *(assert)*
4. Capacidade dos polos reconcilia com a cidade *(assert)* — 4B faz o mesmo com a demanda *(assert)*, 4C garante mentoria ≤ demanda *(assert)*
5 e 6. Distribuição de status por curso, na cidade e no polo

---

**Como usar**: coloque os 7 arquivos de entrada na **mesma pasta deste notebook** — a localização é automática. Depois é só `Kernel → Reiniciar e Executar Tudo`.

**Regras de negócio replicadas e validadas:**
- Filtro de alunos válidos (`DADOSMATRICULAS` ⋈ `BASE_ALUNOS_DW`)
- Carga horária: o notebook confia na base de entrada já corrigida (ver LEIA-ME de `PRATICAS.xlsx`) e apenas avisa — nunca corrige automaticamente — se algum valor parecer bruto
- `CAPACIDADE_LPI`, `QTD_PRATICAS_MODULO` e `FATOR_OCORRENCIAS_MODULO` no resultado final
- `CURSOS_ELEGIVEIS` na aba `RASTREIO_TUTORES_RESUMO`
- **Aba `CURVA_SAVING_POR_MODULO`** (única aba de saving, aberta por módulo):
  - Gráfico 1: CH demandada por tamanho exato de grupo (1, 2, 3... 10+ alunos)
  - Gráfico 2: saving no cenário "só ofertar para grupos com N alunos ou mais" (barras por módulo) **+ linha de TOTAL** (soma de todos os módulos presentes na base — são 3 em 2026, não 4) sobreposta como gráfico combo
- Fórmula de `CH_PRATICAS_TOTAL_TUTOR_POR_DISCIPLINA`: `CH_SEMANAL_PRATICA_DO_TUTOR × QTD_PRATICAS_MODULO`
- Colunas legíveis "Xh Ymin" ao lado de toda métrica de carga horária
- Correção da sinergia HEPROD (auto-referência do curso na própria sinergia)
- Normalização de código de curso (EGRAD_/ESPRE_, HEPROD/EPROD/PROD, EELA/EELE)
- Junção com POLOS (cidade/UF) e DE_PARA (nome de curso/cluster)
- Capacidade do tutor compartilhada entre cidades (não multiplicada por cidade)
- CH de preparação por módulo (multiplicada pelo número de ocorrências)
- Rastreio de tutores com capacidade ociosa (abas `RASTREIO_TUTORES_RESUMO` e `RASTREIO_TUTORES_DETALHE`)
- Hierarquia completa: sinergia → oferta → grupos → demanda → capacidade (tutores) → delta, em todos os níveis

**⚠️ Atenção**: a base `POLOS_INSTITUICAO` tem 124 códigos de unidade com cadastro conflitante. O notebook aplica uma regra automática e gera `RELATORIO_POLOS_CONFLITANTES.csv` na pasta `saida/` para revisão manual.

In [ ]:
import pandas as pd
import numpy as np
import re
import math
import os
import time
import warnings
warnings.filterwarnings('ignore')

t_inicio = time.time()

## CONFIGURAÇÃO — LOCALIZAÇÃO AUTOMÁTICA DOS ARQUIVOS DE ENTRADA Os 7 arquivos de entrada devem estar na MESMA PASTA deste notebook. Não é necessário digitar o nome exato: a busca abaixo procura por um arquivo cujo nome contenha uma das palavras-chave (sem diferenciar maiúsculas/minúsculas ou pequenas variações, como "TUTORES.xlsx" ou "BASE_TUTORES_AR.xlsx"). Se não encontrar, o erro mostra exatamente quais arquivos existem na pasta.

In [ ]:
PASTA_ENTRADA = os.getcwd()  # pasta onde o notebook está sendo executado
PASTA_SAIDA = 'saida'

def _localizar_arquivo(palavras_chave, extensoes, pasta=PASTA_ENTRADA):
    candidatos = [
        arq for arq in os.listdir(pasta)
        if arq.lower().endswith(tuple(extensoes))
        and any(p.lower() in arq.lower() for p in palavras_chave)
    ]
    if not candidatos:
        disponiveis = '\n  '.join(sorted(os.listdir(pasta))) or '(pasta vazia)'
        raise FileNotFoundError(
            f"Não encontrei nenhum arquivo contendo {palavras_chave} "
            f"com extensão {extensoes} em '{pasta}'.\n"
            f"Coloque o arquivo na mesma pasta do notebook. "
            f"Arquivos encontrados nessa pasta:\n  {disponiveis}"
        )
    escolhido = sorted(candidatos)[0]
    if len(candidatos) > 1:
        print(f"[Aviso] Mais de um arquivo bate com {palavras_chave}: {candidatos}. "
              f"Usando '{escolhido}'.")
    return os.path.join(pasta, escolhido)

CAMINHO_DADOSMATRICULAS = _localizar_arquivo(['DADOSMATRICULAS'], ['.csv'])
CAMINHO_BASE_ALUNOS_DW = _localizar_arquivo(['BASE_ALUNOS_DW', 'ALUNOS_DW', 'ALUNOSDW'], ['.csv'])
CAMINHO_POLOS = _localizar_arquivo(['POLOS'], ['.csv'])
CAMINHO_PRATICAS = _localizar_arquivo(['PRATICAS'], ['.xlsx'])
CAMINHO_DE_PARA = _localizar_arquivo(['DE_PARA', 'DEPARA'], ['.xlsx'])
CAMINHO_TUTORES = _localizar_arquivo(['TUTORES'], ['.xlsx'])
CAMINHO_MATRIZES = _localizar_arquivo(['MATRIZES'], ['.csv'])

print("[Configuração] Arquivos de entrada localizados:")
for nome, caminho in [
    ('DADOSMATRICULAS', CAMINHO_DADOSMATRICULAS), ('BASE_ALUNOS_DW', CAMINHO_BASE_ALUNOS_DW),
    ('POLOS', CAMINHO_POLOS), ('PRATICAS', CAMINHO_PRATICAS),
    ('DE_PARA', CAMINHO_DE_PARA), ('TUTORES', CAMINHO_TUTORES),
    ('MATRIZES', CAMINHO_MATRIZES),
]:
    print(f"  {nome:16s} -> {os.path.basename(caminho)}")

PERMITIR_SINERGIA_NSA = True
SEMANAS_DO_MODULO = 10

# =============================================================================
# MODO DE ANÁLISE
#
# 'REALIZADO'  -> demanda vem da matrícula efetiva. Comportamento padrão.
# 'PROJETADO'  -> demanda é a estimativa do módulo seguinte, calculada pela
#                 matriz curricular (Etapa 0B2). A saída vai para um arquivo
#                 separado, para nunca misturar horas medidas com estimadas.
#
# Ordem cíclica dos módulos no ano letivo: 51 -> 52 -> 53 -> 54 -> 51 ...
# MODULO_ORIGEM_PROJECAO: None = último módulo presente na base.
# MODULO_ALVO_PROJECAO:   None = próximo da sequência.
# TAXA_RETENCAO_PROJECAO: None = calcular da base conforme MODO_TAXA_RETENCAO.
# MODO_TAXA_RETENCAO: 'ULTIMA' usa a transição mais recente observada (padrão,
#   reflete a evasão corrente); 'MEDIA' usa a média de todas as transições.
# =============================================================================
MODO_ANALISE = 'REALIZADO'
ORDEM_MODULOS = ['51', '52', '53', '54']
MODULO_ORIGEM_PROJECAO = None
MODULO_ALVO_PROJECAO = None
TAXA_RETENCAO_PROJECAO = None
MODO_TAXA_RETENCAO = 'ULTIMA'

SUFIXO_SAIDA = '' if MODO_ANALISE == 'REALIZADO' else '_PROJETADO'

DE_PARA_CIDADES = {
    'SANTA IZABEL DO PARA': 'SANTA ISABEL DO PARA'
}
DE_PARA_CODIGOS_CURSOS = {
    'HEPROD': 'EPROD',
    'EPROD': 'EPROD',
    'PROD': 'EPROD',
    'EELA': 'EELE',
}

## FUNÇÕES AUXILIARES (equivalentes às funções fx do Power Query)

In [ ]:
_ACENTOS = str.maketrans(
    'ÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ',
    'AAAAAEEEEIIIIOOOOOUUUUC'
)

def fx_normalizar(valor):
    """Maiúsculo, sem acento, sem espaço duplicado, trim."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return ''
    texto = str(valor).strip().upper().translate(_ACENTOS)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def fx_normalizar_serie(serie):
    """Versão vetorizada de fx_normalizar para colunas inteiras do pandas."""
    s = serie.fillna('').astype(str).str.strip().str.upper()
    s = s.str.translate(_ACENTOS)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    return s

def fx_cidade_canonica(valor):
    texto = fx_normalizar(valor)
    return DE_PARA_CIDADES.get(texto, texto)

def fx_codigo_curso_canonico(valor):
    texto = fx_normalizar(valor)
    if texto.startswith('EGRAD_'):
        texto = texto[len('EGRAD_'):]
    elif texto.startswith('ESPRE_'):
        texto = texto[len('ESPRE_'):]
    return DE_PARA_CODIGOS_CURSOS.get(texto, texto)

def fx_corrigir_hora_decimal(valor):
    """1,20 (hora,minuto) -> 1.3333 (hora decimal). Idempotente/seguro."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return valor
    parte_inteira = math.floor(valor)
    parte_minutos = round((valor - parte_inteira) * 100)
    if parte_minutos >= 60:
        return valor
    return parte_inteira + parte_minutos / 60

def fx_cursos_da_sinergia(sinergia):
    """Extrai a lista de códigos canônicos de curso citados na SINERGIA."""
    texto = fx_normalizar(sinergia)
    texto = texto.replace('|', '-')
    partes = [p.strip() for p in texto.split('-')]
    tokens = [p for p in partes if p.startswith('EGRAD_') or p.startswith('ESPRE_')]
    sem_prefixo = [
        t[len('EGRAD_'):] if t.startswith('EGRAD_') else t[len('ESPRE_'):]
        for t in tokens
    ]
    canonicos = sorted(set(fx_codigo_curso_canonico(c) for c in sem_prefixo if c))
    return [c for c in canonicos if c]

def fx_disciplina_oferta(disciplina, sinergia, sinergia_valida):
    disciplina_atual = fx_normalizar(disciplina)
    texto_sinergia = fx_normalizar(sinergia)
    marcador = 'VISTA JUNTO A '
    if sinergia_valida and marcador in texto_sinergia:
        depois = texto_sinergia.split(marcador, 1)[1]
        alvo = depois.split('|')[0]
        return fx_normalizar(alvo)
    return disciplina_atual

def fx_lista_semanas(valor):
    texto = fx_normalizar(valor)
    if texto == '':
        return ['SEM SEMANA']
    if texto == 'NSA':
        return ['NSA']
    partes = [p.strip() for p in texto.split('|') if p.strip() != '']
    return partes if partes else ['SEM SEMANA']

## FORMATAÇÃO LEGÍVEL "Xh Ymin" PARA COLUNAS DE CARGA HORÁRIA As colunas numéricas (horas decimais) continuam existindo — são as usadas em todos os cálculos. Esta função só gera a versão de leitura humana, aplicada como coluna irmã no arquivo final, nunca substituindo o número.

In [ ]:
def fx_formatar_horas_minutos(valor):
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return ''
    sinal = '-' if valor < 0 else ''
    valor_abs = abs(valor)
    horas = int(valor_abs)
    minutos = round((valor_abs - horas) * 60)
    if minutos == 60:
        horas += 1
        minutos = 0
    return f"{sinal}{horas}h {minutos:02d}min"

def adicionar_colunas_legiveis(df, colunas):
    """Insere, ao lado de cada coluna numérica de carga horária, uma coluna
    irmã '<coluna>_HORAS_MINUTOS' em formato 'Xh Ymin'."""
    df = df.copy()
    for col in colunas:
        if col not in df.columns:
            continue
        idx = df.columns.get_loc(col)
        df.insert(idx + 1, f'{col}_HORAS_MINUTOS', df[col].apply(fx_formatar_horas_minutos))
    return df

print(f"[OK] Funções auxiliares carregadas ({time.time()-t_inicio:.1f}s)")

## ETAPA 0A. ALUNOS VÁLIDOS — INNER JOIN DADOSMATRICULAS x BASE_ALUNOS_DW

In [ ]:
t0 = time.time()

dm = pd.read_csv(
    CAMINHO_DADOSMATRICULAS, sep=';', dtype=str,
    usecols=['ALUNO', 'COD_CURSO_ALUNO_ATUAL', 'CURRICULO_ALUNO', 'UNIDADE_FISICA',
             'ANO', 'SEMESTRE', 'DISCIPLINA']
)
dm['ANO'] = dm['ANO'].astype('int16')
dm['SEMESTRE'] = dm['SEMESTRE'].astype('int16')

dw = pd.read_csv(CAMINHO_BASE_ALUNOS_DW, sep=';', dtype=str)
dw['ANO'] = dw['ANO'].astype('int16')
dw['SEMESTRE'] = dw['SEMESTRE'].astype('int16')

alunos_validos = dm.merge(
    dw,
    left_on=['ALUNO', 'COD_CURSO_ALUNO_ATUAL', 'ANO', 'SEMESTRE', 'UNIDADE_FISICA'],
    right_on=['ALUNO', 'COD_CURSO', 'ANO', 'SEMESTRE', 'UNIDADE_FISICA'],
    how='inner'
)

print(f"[Etapa 0A] DADOSMATRICULAS bruto: {len(dm):,} linhas")
print(f"[Etapa 0A] Alunos válidos (após filtro DW): {len(alunos_validos):,} linhas "
      f"({100*len(alunos_validos)/len(dm):.1f}%) — {time.time()-t0:.1f}s")

# CURRICULO_ALUNO alimenta a projeção (Etapa 0B2) — por isso alunos_validos
# só é liberado depois dela.
del dm, dw  # libera memória, não são mais necessários

## ETAPA 0B. CONTAGEM DE ALUNOS POR OFERTA FÍSICA (curso x disciplina x unidade x **módulo**)

O `SEMESTRE` da matrícula entra na chave de agregação e é renomeado para `MODULO`. Sem ele, um aluno matriculado no módulo 51 seria contado também nos módulos 52, 53 e 54 sempre que a mesma disciplina tivesse prática nesses módulos — 43 pares curso-disciplina estão nessa situação.

In [ ]:
t0 = time.time()

# CORREÇÃO: SEMESTRE (= módulo da matrícula) entra na chave de agregação.
# Sem ele, um aluno matriculado no módulo 51 era contado também nos módulos
# 52/53/54 sempre que a mesma disciplina tivesse prática nesses módulos.
demanda_alunos = (
    alunos_validos
    .groupby(['COD_CURSO_ALUNO_ATUAL', 'UNIDADE_FISICA', 'DISCIPLINA', 'SEMESTRE'])
    .size()
    .reset_index(name='QTDE_ALUNOS')
    .rename(columns={'SEMESTRE': 'MODULO'})
)
demanda_alunos['MODULO'] = demanda_alunos['MODULO'].astype(str)

print(f"[Etapa 0B] Ofertas agregadas (curso x disciplina x unidade x módulo): "
      f"{len(demanda_alunos):,} — {time.time()-t0:.1f}s")
print(f"[Etapa 0B] Matrículas contabilizadas: {demanda_alunos['QTDE_ALUNOS'].sum():,}")


## ETAPA 0B2. PROJEÇÃO DO MÓDULO SEGUINTE (só roda com `MODO_ANALISE = 'PROJETADO'`)

Com `MODO_ANALISE = 'REALIZADO'`, esta célula apenas calcula as taxas de retenção observadas, a título de diagnóstico, e não altera nada. Com `'PROJETADO'`, ela **substitui** a demanda realizada pela estimativa do módulo seguinte, e o pipeline inteiro roda em cima dela.

A projeção não usa `SERIE_ALUNO_ATUAL` — esse campo é um snapshot e coincide com a série efetivamente cursada em apenas 41% das linhas. A série vem da matriz: para cada aluno em cada módulo, é a moda do `SERIE_IDEAL` das disciplinas que ele cursa no seu currículo.

Regra: aluno na série S no módulo de origem → série S+1 no módulo alvo, cursando as disciplinas do seu currículo com `SERIE_IDEAL = S+1` e `MODULO = alvo`. Polo e currículo são mantidos. Sobre o total aplica-se a taxa de retenção observada nas transições reais.

In [ ]:
t0 = time.time()

matrizes = pd.read_csv(CAMINHO_MATRIZES, sep=';', dtype=str)
matrizes['SERIE_IDEAL'] = matrizes['SERIE_IDEAL'].astype(int)
matrizes['MODULO'] = matrizes['MODULO'].astype(str)

chave_matriz = matrizes.set_index(['CURRICULO', 'DISCIPLINA'])['SERIE_IDEAL']
chave_matriz = chave_matriz[~chave_matriz.index.duplicated()]

av = alunos_validos[['ALUNO', 'CURRICULO_ALUNO', 'DISCIPLINA', 'SEMESTRE',
                     'COD_CURSO_ALUNO_ATUAL', 'UNIDADE_FISICA']].copy()
av['SEMESTRE'] = av['SEMESTRE'].astype(str)
av = av.join(chave_matriz.rename('SERIE_MATRIZ'), on=['CURRICULO_ALUNO', 'DISCIPLINA'])
print(f"[Etapa 0B2] Matrículas com disciplina localizada na matriz do currículo: "
      f"{av['SERIE_MATRIZ'].notna().mean():.1%}")

av_ok = av[av['SERIE_MATRIZ'].notna()]
serie_cursada = (
    av_ok.groupby(['ALUNO', 'SEMESTRE', 'CURRICULO_ALUNO',
                   'COD_CURSO_ALUNO_ATUAL', 'UNIDADE_FISICA'])['SERIE_MATRIZ']
    .agg(lambda s: s.mode().iloc[0])
    .reset_index()
    .rename(columns={'SERIE_MATRIZ': 'SERIE_CURSADA', 'SEMESTRE': 'MODULO'})
)

# Diagnóstico: retenção e avanço de série observados entre módulos.
pivot_serie = serie_cursada.pivot_table(
    index='ALUNO', columns='MODULO', values='SERIE_CURSADA', aggfunc='max'
)
modulos_presentes = [m for m in ORDEM_MODULOS if m in pivot_serie.columns]
retencoes = []
for anterior, seguinte in zip(modulos_presentes, modulos_presentes[1:]):
    par = pivot_serie[[anterior, seguinte]].dropna()
    base_anterior = pivot_serie[anterior].notna().sum()
    if base_anterior == 0:
        continue
    retencao = len(par) / base_anterior
    retencoes.append(retencao)
    print(f"[Etapa 0B2] Transição {anterior}->{seguinte}: retenção {retencao:.1%} | "
          f"série avançou +1 em {(par[seguinte] == par[anterior] + 1).mean():.1%} "
          f"dos que seguiram")

if TAXA_RETENCAO_PROJECAO is not None:
    taxa_retencao, origem_taxa = TAXA_RETENCAO_PROJECAO, 'fixada manualmente'
elif not retencoes:
    taxa_retencao, origem_taxa = 1.0, 'sem transição observada — projeção sem evasão'
elif MODO_TAXA_RETENCAO == 'MEDIA':
    taxa_retencao = float(np.mean(retencoes))
    origem_taxa = f'média de {len(retencoes)} transições observadas'
else:
    taxa_retencao, origem_taxa = float(retencoes[-1]), 'última transição observada'

if MODO_ANALISE != 'PROJETADO':
    print("[Etapa 0B2] MODO_ANALISE = 'REALIZADO' — demanda inalterada. "
          f"(Taxa de retenção disponível para projeção: {taxa_retencao:.1%}.)")
else:
    modulo_origem = MODULO_ORIGEM_PROJECAO or modulos_presentes[-1]
    modulo_alvo = MODULO_ALVO_PROJECAO or ORDEM_MODULOS[
        (ORDEM_MODULOS.index(modulo_origem) + 1) % len(ORDEM_MODULOS)
    ]

    base_origem = serie_cursada[serie_cursada['MODULO'] == modulo_origem].copy()
    base_origem['SERIE_ALVO'] = base_origem['SERIE_CURSADA'] + 1

    disciplinas_alvo = matrizes[matrizes['MODULO'] == modulo_alvo][
        ['CURRICULO', 'DISCIPLINA', 'SERIE_IDEAL']
    ].drop_duplicates()

    projecao = base_origem.merge(
        disciplinas_alvo,
        left_on=['CURRICULO_ALUNO', 'SERIE_ALVO'],
        right_on=['CURRICULO', 'SERIE_IDEAL'],
        how='inner'
    )
    print(f"[Etapa 0B2] Projetando {modulo_origem} -> {modulo_alvo} | "
          f"alunos na origem: {len(base_origem):,} | com disciplina prevista no alvo: "
          f"{projecao['ALUNO'].nunique():,}")
    print(f"[Etapa 0B2] Taxa de retenção aplicada: {taxa_retencao:.1%} ({origem_taxa})")

    demanda_alunos = (
        projecao
        .groupby(['COD_CURSO_ALUNO_ATUAL', 'UNIDADE_FISICA', 'DISCIPLINA'])
        .size()
        .reset_index(name='QTDE_ALUNOS')
    )
    demanda_alunos['QTDE_ALUNOS'] = (
        (demanda_alunos['QTDE_ALUNOS'] * taxa_retencao).round().astype(int)
    )
    demanda_alunos = demanda_alunos[demanda_alunos['QTDE_ALUNOS'] > 0].copy()
    demanda_alunos['MODULO'] = modulo_alvo

    print(f"[Etapa 0B2] Demanda projetada: {len(demanda_alunos):,} ofertas | "
          f"{demanda_alunos['QTDE_ALUNOS'].sum():,} matrículas esperadas")

del alunos_validos, av, av_ok
print(f"[Etapa 0B2] Concluída — {time.time()-t0:.1f}s")


## ETAPA 0C. BASE DE PRÁTICAS: correção de hora + junção com demanda de alunos

A junção com a demanda usa `COD_CURSO + COD_DISCIPLINA + MODULO`. O módulo é parte obrigatória da chave: sem ele, uma disciplina com prática em dois módulos recebe o contingente inteiro de alunos nos dois.

In [ ]:
t0 = time.time()

praticas = pd.read_excel(CAMINHO_PRATICAS, sheet_name='BASE_PRATICAS')

# A base de entrada já vem corrigida na origem (ver aba LEIA-ME do
# PRATICAS.xlsx). O notebook NÃO reaplica a correção automaticamente — um
# valor já certo como 1,333333 (vindo de 1h20min) tem uma parte decimal
# (",33") que a mesma heurística reinterpretaria como "33 minutos",
# corrompendo-o para 1,55h. Em vez disso, só AVISA se algum valor ainda
# parecer estar no formato bruto "hora,minuto" (2 casas decimais "limpas",
# sem a dízima característica de uma conversão já feita).
def _parece_hora_bruta_nao_corrigida(valor):
    if pd.isna(valor):
        return False
    fracionaria = valor - math.floor(valor)
    if fracionaria < 1e-9:
        return False
    return abs(round(fracionaria, 2) - round(fracionaria, 6)) < 1e-9

for _col in ['CH_SEMANAL_DURACAO_DA_PRATICA', 'CH_SEMANAL_PRATICA_DO_TUTOR']:
    _suspeitos = praticas[praticas[_col].apply(_parece_hora_bruta_nao_corrigida)]
    if len(_suspeitos) > 0:
        print(f"[AVISO] {len(_suspeitos)} linha(s) em {_col} parecem estar no formato "
              f"bruto 'hora,minuto' (valores: {sorted(_suspeitos[_col].unique())}). "
              f"Corrija a base de origem — o notebook não corrige automaticamente "
              f"para não arriscar corromper valores que já estejam certos.")

# CH_PRATICAS_TOTAL_TUTOR_POR_DISCIPLINA = CH_SEMANAL_PRATICA_DO_TUTOR x
# QTD_PRATICAS_MODULO (validado em toda a base: é a CH do TUTOR, não a
# duração da prática — as duas só coincidem quando os dois valores são
# iguais na linha, o que não é sempre o caso, ex.: Bioquímica Clínica tem
# 3h de duração mas 4h de CH do tutor).
praticas['CH_PRATICAS_TOTAL_TUTOR_POR_DISCIPLINA'] = (
    praticas['CH_SEMANAL_PRATICA_DO_TUTOR'] * praticas['QTD_PRATICAS_MODULO']
)
praticas['MODULO'] = praticas['MODULO'].astype(str)

# INNER JOIN: só entram práticas que têm alunos reais em alguma unidade
# CORREÇÃO: MODULO faz parte da chave. Sem ele, uma disciplina com prática
# em dois módulos recebia o contingente inteiro de alunos nos dois.
praticas_com_alunos = praticas.merge(
    demanda_alunos,
    left_on=['COD_CURSO', 'COD_DISCIPLINA', 'MODULO'],
    right_on=['COD_CURSO_ALUNO_ATUAL', 'DISCIPLINA', 'MODULO'],
    how='inner'
)

# Quantidade de grupos "isolada" (sem considerar sinergia) — usada como
# comparação para mostrar quanto a sinergia economizou.
praticas_com_alunos['QTDE_GRUPOS'] = np.where(
    (praticas_com_alunos['CAPACIDADE_LPI'].isna()) | (praticas_com_alunos['CAPACIDADE_LPI'] <= 0),
    np.nan,
    np.ceil(praticas_com_alunos['QTDE_ALUNOS'] / praticas_com_alunos['CAPACIDADE_LPI'])
)

print(f"[Etapa 0C] Práticas base: {len(praticas):,} linhas")
print(f"[Etapa 0C] Após expandir por oferta com alunos: {len(praticas_com_alunos):,} "
      f"— {time.time()-t0:.1f}s")

# =============================================================================
# ETAPA 0D. POLOS — cidade/UF/nome do polo.
# IMPORTANTE: UNIDADE_FISICA nunca tem 4 dígitos com zero à esquerda (validado
# nos dados reais), então NÃO aplicamos padding — ele colidiria "117" com o
# código real "0117" (série SEDE, que é DIFERENTE). O match é direto, como texto.
#
# A base POLOS também tem 124 códigos de unidade com cadastro CONFLITANTE
# (mesma COD_UNIDADE, cidades diferentes — ex.: "1329" aparece como
# "MACARANI-BA" e também "MARINGÁ-PR"). Isso é um problema de qualidade de
# dados na origem, não algo que dá pra resolver com certeza aqui. Regra
# aplicada: preferir a linha com cidade cadastrada (não "NÃO CADASTRADO");
# em empate, manter a primeira ocorrência. Todos os códigos afetados são
# exportados em RELATORIO_POLOS_CONFLITANTES.csv para revisão manual.
# =============================================================================
t0 = time.time()

polos = pd.read_csv(CAMINHO_POLOS, sep=';', dtype=str)

polos_dedup = polos[['COD_UNIDADE', 'NM_UNIDADE', 'NM_CIDADE', 'CD_UF']].drop_duplicates()
polos_dedup['_prioridade'] = (polos_dedup['NM_CIDADE'] == 'NÃO CADASTRADO').astype(int)
polos_dedup = polos_dedup.sort_values(['COD_UNIDADE', '_prioridade'])

codigos_conflitantes = polos_dedup[
    polos_dedup.duplicated(subset=['COD_UNIDADE'], keep=False)
]['COD_UNIDADE'].unique()

if len(codigos_conflitantes) > 0:
    relatorio_conflitos = polos[polos['COD_UNIDADE'].isin(codigos_conflitantes)].sort_values('COD_UNIDADE')
    os.makedirs(PASTA_SAIDA, exist_ok=True)
    relatorio_conflitos.to_csv(f'{PASTA_SAIDA}/RELATORIO_POLOS_CONFLITANTES.csv', sep=';', index=False)
    print(f"[Etapa 0D] ATENÇÃO: {len(codigos_conflitantes)} códigos de unidade com cadastro "
          f"conflitante em POLOS. Relatório salvo em {PASTA_SAIDA}/RELATORIO_POLOS_CONFLITANTES.csv "
          f"para revisão manual.")

polos_final = (
    polos_dedup.drop_duplicates(subset=['COD_UNIDADE'], keep='first')
    .drop(columns='_prioridade')
)

praticas_com_alunos['UNIDADE_FISICA'] = praticas_com_alunos['UNIDADE_FISICA'].str.strip()
polos_final['COD_UNIDADE'] = polos_final['COD_UNIDADE'].str.strip()

fonte_consolidada = praticas_com_alunos.merge(
    polos_final[['COD_UNIDADE', 'NM_UNIDADE', 'NM_CIDADE', 'CD_UF']],
    left_on='UNIDADE_FISICA', right_on='COD_UNIDADE', how='left'
)

sem_cidade = fonte_consolidada['NM_CIDADE'].isna().sum()
print(f"[Etapa 0D] Linhas sem cidade correspondente: {sem_cidade} de {len(fonte_consolidada):,} "
      f"— {time.time()-t0:.1f}s")
assert len(fonte_consolidada) == len(praticas_com_alunos), \
    "O join com POLOS multiplicou linhas — ainda há duplicidade de COD_UNIDADE!"

## ETAPA 0E. DE_PARA — nome do curso, cluster e sigla padrão (código canônico)

In [ ]:
t0 = time.time()

de_para = pd.read_excel(CAMINHO_DE_PARA, sheet_name='DE_PARA')

fonte_consolidada = fonte_consolidada.merge(
    de_para[['COD_CURSO', 'NOME_CURSO', 'CLUSTER', 'SIGLA_CURSO_PADRAO']],
    on='COD_CURSO', how='left'
)
fonte_consolidada = fonte_consolidada.rename(columns={'SIGLA_CURSO_PADRAO': 'CURSO_AGRUPADO'})

sem_curso = fonte_consolidada['NOME_CURSO'].isna().sum()
print(f"[Etapa 0E] Linhas sem nome de curso correspondente: {sem_curso} "
      f"— {time.time()-t0:.1f}s")

# Ajustes finais de nomes/tipos para entrar na cadeia de cálculo hierárquico
fonte_consolidada['DISCIPLINA_NORMALIZADA'] = fonte_consolidada['NOME_DISCIPLINA']
fonte_consolidada['UNIDADE_FISICA'] = fonte_consolidada['UNIDADE_FISICA'].str.lstrip('0')
fonte_consolidada.loc[fonte_consolidada['UNIDADE_FISICA'] == '', 'UNIDADE_FISICA'] = '0'

print(f"\n[Etapa 0] Total: {len(fonte_consolidada):,} linhas — "
      f"{time.time()-t_inicio:.1f}s desde o início")

## 1. VALIDAÇÃO DE SINERGIA

In [ ]:
t0 = time.time()

fc = fonte_consolidada  # apelido curto

fc['CIDADE_CHAVE'] = fc['NM_CIDADE'].apply(fx_cidade_canonica)
fc['CURSO_CHAVE'] = fc['NOME_CURSO'].apply(fx_normalizar)
fc['CURSO_CODIGO_CANONICO'] = fc['CURSO_AGRUPADO'].apply(fx_codigo_curso_canonico)
fc['CURSOS_SINERGIA_LISTA'] = fc['SINERGIA'].apply(fx_cursos_da_sinergia)

# Inclui automaticamente o código canônico do próprio curso na lista da
# sinergia quando SIM| está declarado — corrige o caso em que a string de
# sinergia lista os outros cursos mas omite a si mesma (bug do HEPROD já
# corrigido na origem, mantido aqui como trava de segurança adicional).
def _completar_lista_sinergia(row):
    sinergia_norm = fx_normalizar(row['SINERGIA'])
    base = row['CURSOS_SINERGIA_LISTA']
    if 'SIM|' in sinergia_norm and len(base) > 0:
        return sorted(set(base) | {row['CURSO_CODIGO_CANONICO']})
    return base

fc['CURSOS_SINERGIA_LISTA'] = fc.apply(_completar_lista_sinergia, axis=1)

fc['SEMANA_ANALISE'] = fc['SEMANA_DA_PRATICA'].apply(fx_lista_semanas)
fc = fc.explode('SEMANA_ANALISE', ignore_index=True)

fc['SINERGIA_NORM'] = fc['SINERGIA'].apply(fx_normalizar)
fc['SINERGIA_BASE_VALIDA'] = (
    fc['SINERGIA_NORM'].str.contains('SIM|', regex=False)
    & (fc['CURSOS_SINERGIA_LISTA'].apply(len) >= 2)
    & fc.apply(lambda r: r['CURSO_CODIGO_CANONICO'] in r['CURSOS_SINERGIA_LISTA'], axis=1)
)
fc['SINERGIA_VALIDA'] = fc['SINERGIA_BASE_VALIDA'] & (
    (fc['SEMANA_ANALISE'] != 'NSA') | PERMITIR_SINERGIA_NSA
)

def _motivo_sinergia(row):
    if 'SIM|' not in row['SINERGIA_NORM']:
        return 'SEM SINERGIA DECLARADA'
    if len(row['CURSOS_SINERGIA_LISTA']) < 2:
        return 'SEM SINERGIA ENTRE CURSOS'
    if row['CURSO_CODIGO_CANONICO'] not in row['CURSOS_SINERGIA_LISTA']:
        return 'SINERGIA DESCARTADA - CURSO NAO PARTICIPANTE'
    if row['SEMANA_ANALISE'] == 'NSA' and not PERMITIR_SINERGIA_NSA:
        return 'SINERGIA DESCARTADA - NSA BLOQUEADO'
    if row['SEMANA_ANALISE'] == 'NSA':
        return 'SINERGIA VALIDA DECLARADA EM NSA'
    return 'SINERGIA VALIDA'

fc['MOTIVO_REGRA_SINERGIA'] = fc.apply(_motivo_sinergia, axis=1)

fc['DISCIPLINA_OFERTA'] = fc.apply(
    lambda r: fx_disciplina_oferta(r['DISCIPLINA_NORMALIZADA'], r['SINERGIA'], r['SINERGIA_VALIDA']),
    axis=1
)

fc['CHAVE_CURSOS_OFERTA'] = np.where(
    fc['SINERGIA_VALIDA'],
    'SINERGIA:' + fc['CURSOS_SINERGIA_LISTA'].apply(lambda l: '|'.join(l)),
    'CURSO:' + fc['CURSO_CHAVE']
)

# Linhas válidas: cidade, curso e disciplina precisam existir
linhas_validas = fc[
    (fc['CIDADE_CHAVE'] != '') & (fc['CURSO_CHAVE'] != '') & (fc['DISCIPLINA_OFERTA'] != '')
].copy()

print(f"[Etapa 1] Linhas válidas (com semana explodida): {len(linhas_validas):,} "
      f"— {time.time()-t0:.1f}s")

## 2. OFERTA REAL: CURSOS SÓ SE UNEM QUANDO SINERGIA_VALIDA = TRUE

In [ ]:
t0 = time.time()

CHAVE_OFERTA = [
    'CIDADE_CHAVE', 'UNIDADE_FISICA', 'CLUSTER', 'MODULO', 'SEMANA_ANALISE',
    'DISCIPLINA_OFERTA', 'CHAVE_CURSOS_OFERTA', 'CAPACIDADE_LPI',
    'QTD_PRATICAS_MODULO', 'CH_SEMANAL_DURACAO_DA_PRATICA',
    'CH_SEMANAL_PRATICA_DO_TUTOR', 'CH_PRATICAS_TOTAL_TUTOR_POR_DISCIPLINA'
]

def _primeiro_texto(lista):
    limpos = sorted(set(str(x).strip() for x in lista if pd.notna(x) and str(x).strip() != ''))
    return limpos[0] if limpos else None

ofertas_agrupadas = linhas_validas.groupby(CHAVE_OFERTA, dropna=False).agg(
    NM_CIDADE_OFERTA=('NM_CIDADE', _primeiro_texto),
    QTDE_CURSOS_PRESENTES_OFERTA=('CURSO_CHAVE', lambda s: s.nunique()),
    QTDE_ALUNOS_OFERTA=('QTDE_ALUNOS', 'sum'),
    QTDE_GRUPOS_ANTES_SINERGIA=('QTDE_GRUPOS', 'sum'),
    TEM_LINHA_SINERGIA_VALIDA=('SINERGIA_VALIDA', 'any'),
).reset_index()

ofertas_agrupadas = ofertas_agrupadas.sort_values(CHAVE_OFERTA).reset_index(drop=True)
ofertas_agrupadas['INDICE_OFERTA'] = ofertas_agrupadas.index + 1
ofertas_agrupadas['ID_OFERTA'] = 'OFERTA-' + ofertas_agrupadas['INDICE_OFERTA'].astype(str).str.zfill(8)

ofertas_agrupadas['SINERGIA_APLICADA'] = (
    ofertas_agrupadas['TEM_LINHA_SINERGIA_VALIDA'] & (ofertas_agrupadas['QTDE_CURSOS_PRESENTES_OFERTA'] >= 2)
)

ofertas_agrupadas['QTDE_GRUPOS_OFERTA'] = np.where(
    (ofertas_agrupadas['CAPACIDADE_LPI'].isna()) | (ofertas_agrupadas['CAPACIDADE_LPI'] <= 0),
    np.nan,
    np.ceil(ofertas_agrupadas['QTDE_ALUNOS_OFERTA'] / ofertas_agrupadas['CAPACIDADE_LPI'])
)
ofertas_agrupadas['GRUPOS_REDUZIDOS_SINERGIA'] = (
    ofertas_agrupadas['QTDE_GRUPOS_ANTES_SINERGIA'] - ofertas_agrupadas['QTDE_GRUPOS_OFERTA']
)
ofertas_agrupadas['DEMANDA_CH_OFERTA'] = (
    ofertas_agrupadas['QTDE_GRUPOS_OFERTA'] * ofertas_agrupadas['CH_SEMANAL_PRATICA_DO_TUTOR']
)
ofertas_agrupadas['CH_PREPARACAO_OFERTA'] = (
    ofertas_agrupadas['QTDE_GRUPOS_OFERTA']
    * (ofertas_agrupadas['CH_SEMANAL_PRATICA_DO_TUTOR'] - ofertas_agrupadas['CH_SEMANAL_DURACAO_DA_PRATICA']).clip(lower=0)
)

def _status_oferta(row):
    if row['SINERGIA_APLICADA'] and row['SEMANA_ANALISE'] == 'NSA':
        return 'SINERGIA APLICADA - DECLARADA EM NSA'
    if row['SINERGIA_APLICADA']:
        return 'SINERGIA APLICADA'
    if row['TEM_LINHA_SINERGIA_VALIDA']:
        return 'SINERGIA PREVISTA - SOMENTE UM CURSO PRESENTE'
    return 'SEM SINERGIA - CURSO MANTIDO SEPARADO'

ofertas_agrupadas['STATUS_SINERGIA'] = ofertas_agrupadas.apply(_status_oferta, axis=1)

print(f"[Etapa 2] Ofertas únicas (após sinergia): {len(ofertas_agrupadas):,} "
      f"— {time.time()-t0:.1f}s")
print(f"[Etapa 2] Ofertas com sinergia aplicada: {ofertas_agrupadas['SINERGIA_APLICADA'].sum():,}")
print(f"[Etapa 2] Grupos economizados pela sinergia: "
      f"{ofertas_agrupadas['GRUPOS_REDUZIDOS_SINERGIA'].sum():,.0f}")

## 3. ABERTURA CURSO E DISCIPLINA DENTRO DA OFERTA

In [ ]:
t0 = time.time()

CHAVE_DETALHE = CHAVE_OFERTA + [
    'CURSO_CHAVE', 'CURSO_CODIGO_CANONICO', 'CURSO_AGRUPADO', 'NOME_CURSO',
    'DISCIPLINA_NORMALIZADA', 'NOME_DISCIPLINA'
]

detalhe = linhas_validas.groupby(CHAVE_DETALHE, dropna=False).agg(
    NM_CIDADE=('NM_CIDADE', _primeiro_texto),
    NM_UNIDADE=('NM_UNIDADE', _primeiro_texto),
    CD_UF=('CD_UF', _primeiro_texto),
    QTDE_ALUNOS_LINHA=('QTDE_ALUNOS', 'sum'),
    QTDE_GRUPOS_CURSO_ISOLADO=('QTDE_GRUPOS', 'sum'),
    MOTIVO_REGRA_SINERGIA=('MOTIVO_REGRA_SINERGIA', lambda s: ' | '.join(sorted(set(s)))),
    SINERGIA_VALIDA_LINHA=('SINERGIA_VALIDA', 'any'),
).reset_index()

detalhe = detalhe.merge(
    ofertas_agrupadas[CHAVE_OFERTA + [
        'ID_OFERTA', 'QTDE_CURSOS_PRESENTES_OFERTA', 'QTDE_ALUNOS_OFERTA',
        'QTDE_GRUPOS_ANTES_SINERGIA', 'QTDE_GRUPOS_OFERTA', 'GRUPOS_REDUZIDOS_SINERGIA',
        'DEMANDA_CH_OFERTA', 'CH_PREPARACAO_OFERTA', 'SINERGIA_APLICADA', 'STATUS_SINERGIA'
    ]],
    on=CHAVE_OFERTA, how='left'
)

detalhe['PESO_ALUNOS_NA_OFERTA'] = np.where(
    (detalhe['QTDE_ALUNOS_OFERTA'].isna()) | (detalhe['QTDE_ALUNOS_OFERTA'] == 0),
    0,
    detalhe['QTDE_ALUNOS_LINHA'] / detalhe['QTDE_ALUNOS_OFERTA']
)
detalhe['DEMANDA_CH_AUX_RATEIO_CURSO'] = detalhe['DEMANDA_CH_OFERTA'] * detalhe['PESO_ALUNOS_NA_OFERTA']
detalhe['CH_PREPARACAO_AUX_RATEIO_CURSO'] = detalhe['CH_PREPARACAO_OFERTA'] * detalhe['PESO_ALUNOS_NA_OFERTA']

# Uma única linha "contabilizadora" por oferta — evita duplicar grupos físicos
# quando a oferta reúne mais de um curso via sinergia.
detalhe = detalhe.sort_values(['ID_OFERTA', 'NOME_CURSO', 'DISCIPLINA_NORMALIZADA']).reset_index(drop=True)
detalhe['ORDEM_LINHA_OFERTA'] = detalhe.groupby('ID_OFERTA').cumcount() + 1
detalhe['LINHA_CONTABILIZADORA_OFERTA'] = detalhe['ORDEM_LINHA_OFERTA'] == 1

detalhe['QTDE_GRUPOS_RATEADOS_LINHA'] = np.where(detalhe['LINHA_CONTABILIZADORA_OFERTA'], detalhe['QTDE_GRUPOS_OFERTA'], 0)
detalhe['DEMANDA_CH_RATEADA_LINHA'] = np.where(detalhe['LINHA_CONTABILIZADORA_OFERTA'], detalhe['DEMANDA_CH_OFERTA'], 0)
detalhe['CH_PREPARACAO_RATEADA_LINHA'] = np.where(detalhe['LINHA_CONTABILIZADORA_OFERTA'], detalhe['CH_PREPARACAO_OFERTA'], 0)
detalhe['QTDE_GRUPOS_CONTABILIZADOS'] = detalhe['QTDE_GRUPOS_RATEADOS_LINHA']
detalhe['DEMANDA_CH_CONTABILIZADA'] = detalhe['DEMANDA_CH_RATEADA_LINHA']

# Fator de ocorrências no módulo: NSA/SEM SEMANA usa QTD_PRATICAS_MODULO
# (representa todas as ocorrências); semanas explícitas contam 1 vez.
detalhe['FATOR_OCORRENCIAS_MODULO'] = np.where(
    detalhe['SEMANA_ANALISE'].isin(['NSA', 'SEM SEMANA']),
    detalhe['QTD_PRATICAS_MODULO'].fillna(1).replace(0, 1),
    1
)
detalhe['DEMANDA_CH_MODULO_LINHA'] = detalhe['DEMANDA_CH_RATEADA_LINHA'] * detalhe['FATOR_OCORRENCIAS_MODULO']
detalhe['DEMANDA_CH_MODULO_AUX_RATEIO_CURSO'] = detalhe['DEMANDA_CH_AUX_RATEIO_CURSO'] * detalhe['FATOR_OCORRENCIAS_MODULO']

# CH de preparação também precisa ser multiplicada pelo fator de ocorrências
# do módulo: se a prática acontece N vezes no módulo (QTD_PRATICAS_MODULO em
# NSA/SEM SEMANA, ou 1x por semana explícita), a preparação se repete a cada
# ocorrência. Antes, só existia a versão "por ocorrência" (CH_PREPARACAO_
# RATEADA_LINHA); esta é a versão consolidada no módulo inteiro.
detalhe['CH_PREPARACAO_MODULO_LINHA'] = detalhe['CH_PREPARACAO_RATEADA_LINHA'] * detalhe['FATOR_OCORRENCIAS_MODULO']

print(f"[Etapa 3] Linhas de detalhe (curso x disciplina dentro da oferta): {len(detalhe):,} "
      f"— {time.time()-t0:.1f}s")

## 3B. GRUPOS FÍSICOS E ALUNOS ÚNICOS NO MÓDULO (sem duplicar entre semanas)

In [ ]:
t0 = time.time()

CHAVE_GRUPO_FISICO_MODULO = [c for c in CHAVE_OFERTA if c != 'SEMANA_ANALISE']

resumo_grupo_modulo = detalhe.groupby(CHAVE_GRUPO_FISICO_MODULO, dropna=False).agg(
    ID_OFERTA_CONTABILIZADORA_GRUPO_MODULO=('ID_OFERTA', 'min'),
    QTDE_GRUPOS_FISICOS_MODULO=('QTDE_GRUPOS_OFERTA', lambda s: s.max() if s.notna().any() else 0),
).reset_index()

detalhe = detalhe.merge(resumo_grupo_modulo, on=CHAVE_GRUPO_FISICO_MODULO, how='left')
detalhe['LINHA_CONTABILIZADORA_GRUPO_MODULO'] = (
    (detalhe['ID_OFERTA'] == detalhe['ID_OFERTA_CONTABILIZADORA_GRUPO_MODULO'])
    & detalhe['LINHA_CONTABILIZADORA_OFERTA']
)
detalhe['QTDE_GRUPOS_FISICOS_MODULO_LINHA'] = np.where(
    detalhe['LINHA_CONTABILIZADORA_GRUPO_MODULO'], detalhe['QTDE_GRUPOS_FISICOS_MODULO'], 0
)
detalhe['QTDE_GRUPOS_ATENDIMENTOS_MODULO_LINHA'] = detalhe['QTDE_GRUPOS_FISICOS_MODULO_LINHA']

CHAVE_ALUNOS_MODULO = ['CIDADE_CHAVE', 'UNIDADE_FISICA', 'MODULO', 'CURSO_CHAVE', 'DISCIPLINA_NORMALIZADA']
detalhe['ID_LINHA_DETALHE'] = detalhe['ID_OFERTA'] + '|' + detalhe['ORDEM_LINHA_OFERTA'].astype(str).str.zfill(6)

resumo_alunos_modulo = detalhe.groupby(CHAVE_ALUNOS_MODULO, dropna=False).agg(
    ID_LINHA_CONTABILIZADORA_ALUNOS_MODULO=('ID_LINHA_DETALHE', 'min'),
    QTDE_ALUNOS_MODULO=('QTDE_ALUNOS_LINHA', lambda s: s.max() if s.notna().any() else 0),
).reset_index()

detalhe = detalhe.merge(resumo_alunos_modulo, on=CHAVE_ALUNOS_MODULO, how='left')
detalhe['LINHA_CONTABILIZADORA_ALUNOS_MODULO'] = (
    detalhe['ID_LINHA_DETALHE'] == detalhe['ID_LINHA_CONTABILIZADORA_ALUNOS_MODULO']
)
detalhe['QTDE_ALUNOS_MODULO_LINHA'] = np.where(
    detalhe['LINHA_CONTABILIZADORA_ALUNOS_MODULO'], detalhe['QTDE_ALUNOS_MODULO'], 0
)

print(f"[Etapa 3B] Grupos físicos e alunos únicos no módulo calculados — {time.time()-t0:.1f}s")

## 4. DEMANDA POR CURSO (para vincular somente tutores contratados no curso)

In [ ]:
t0 = time.time()

demanda_curso = detalhe.groupby(['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE', 'CURSO_CHAVE'], dropna=False).agg(
    NOME_CURSO_DEMANDA=('NOME_CURSO', _primeiro_texto),
    DEMANDA_CH_CURSO=('DEMANDA_CH_AUX_RATEIO_CURSO', 'sum'),
    QTDE_ALUNOS_CURSO=('QTDE_ALUNOS_LINHA', 'sum'),
).reset_index()

demanda_modulo_curso = detalhe.groupby(['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE'], dropna=False).agg(
    NOME_CURSO_DEMANDA_MODULO=('NOME_CURSO', _primeiro_texto),
    DEMANDA_CH_MODULO_CURSO=('DEMANDA_CH_MODULO_AUX_RATEIO_CURSO', 'sum'),
).reset_index()

print(f"[Etapa 4] Demanda por curso calculada — {time.time()-t0:.1f}s")

## 5. TUTORES: MATRÍCULA + CIDADE + CURSO

In [ ]:
t0 = time.time()

tutores = pd.read_excel(CAMINHO_TUTORES, sheet_name='BASE_TUTORES')
tutores['CIDADE_CHAVE'] = tutores['CIDADE'].apply(fx_cidade_canonica)
tutores['CURSO_CHAVE'] = tutores['CURSO'].apply(fx_normalizar)
tutores['MATRICULA'] = tutores['MATRICULA'].astype(str)

tutores_validos = tutores[
    (tutores['CIDADE_CHAVE'] != '') & (tutores['CURSO_CHAVE'] != '')
    & (tutores['MATRICULA'].str.strip() != '')
    & (~tutores['CURSO_CHAVE'].isin(['NAO INFORMADO', 'SEM INFORMACAO']))
].copy()

tutor_curso_cidade = tutores_validos.groupby(['CIDADE_CHAVE', 'CURSO_CHAVE', 'MATRICULA'], dropna=False).agg(
    NOME_TUTOR=('NOME', _primeiro_texto),
    CH_SEMANAL=('CH SEMANAL', lambda s: s.max() if s.notna().any() else 0),
    QTDE_VALORES_CH=('CH SEMANAL', lambda s: s.dropna().nunique()),
).reset_index()

print(f"[Etapa 5] Tutores válidos únicos (matrícula x curso x cidade): {len(tutor_curso_cidade):,} "
      f"— {time.time()-t0:.1f}s")

## 5A. CAPACIDADE ACUMULADA DO MÓDULO POR CURSO ELEGÍVEL

In [ ]:
t0 = time.time()

tutores_demanda_modulo = tutor_curso_cidade.merge(
    demanda_modulo_curso, on=['CIDADE_CHAVE', 'CURSO_CHAVE'], how='inner'
)

tutor_unico_modulo = tutores_demanda_modulo.groupby(['CIDADE_CHAVE', 'MATRICULA', 'MODULO'], dropna=False).agg(
    NOME_TUTOR=('NOME_TUTOR', _primeiro_texto),
    CH_SEMANAL_TUTOR=('CH_SEMANAL', lambda s: s.max() if s.notna().any() else 0),
).reset_index()

resumo_tutores_cidade_modulo = tutor_unico_modulo.groupby(['CIDADE_CHAVE', 'MODULO'], dropna=False).agg(
    QTDE_TUTORES_MODULO_CIDADE_ELEGIVEIS=('MATRICULA', 'count'),
    CH_SEMANAL_TUTORES_MODULO_CIDADE_ELEGIVEIS=('CH_SEMANAL_TUTOR', 'sum'),
).reset_index()
resumo_tutores_cidade_modulo['CH_CAPACIDADE_TOTAL_MODULO_TUTORES_ELEGIVEIS'] = (
    resumo_tutores_cidade_modulo['CH_SEMANAL_TUTORES_MODULO_CIDADE_ELEGIVEIS'] * SEMANAS_DO_MODULO
)

def _rateio_vetorizado(df, chave_grupo, col_demanda, col_ch_tutor, col_saida):
    """Rateia a capacidade do tutor entre os cursos, proporcional à demanda
    de cada curso dentro do grupo (cidade+matrícula[+módulo/semana]).
    Versão vetorizada com groupby().transform() — muito mais rápida que
    apply() linha a linha em bases grandes."""
    demanda_elegivel = df.groupby(chave_grupo)[col_demanda].transform('sum')
    ch_tutor = df.groupby(chave_grupo)[col_ch_tutor].transform('max')
    capacidade_tutor = ch_tutor * SEMANAS_DO_MODULO
    df[col_saida] = np.where(
        demanda_elegivel == 0, 0, capacidade_tutor * df[col_demanda] / demanda_elegivel
    )
    return df

distrib_tutor_modulo = _rateio_vetorizado(
    tutores_demanda_modulo.copy(),
    ['MATRICULA', 'MODULO'],  # capacidade é da PESSOA, não de tutor+cidade — um tutor
                               # elegível em várias cidades tem uma capacidade só, compartilhada
    'DEMANDA_CH_MODULO_CURSO', 'CH_SEMANAL',
    'CH_CAPACIDADE_MODULO_RATEADA_TUTOR_CURSO'
)

capacidade_modulo_curso = distrib_tutor_modulo.groupby(['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE'], dropna=False).agg(
    QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO=('MATRICULA', 'nunique'),
    CH_CAPACIDADE_MODULO_RATEADA_CURSO=('CH_CAPACIDADE_MODULO_RATEADA_TUTOR_CURSO', 'sum'),
).reset_index()

resumo_modulo_curso = demanda_modulo_curso.merge(
    capacidade_modulo_curso, on=['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE'], how='left'
)
resumo_modulo_curso['QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO'] = resumo_modulo_curso['QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO'].fillna(0)
resumo_modulo_curso['CH_CAPACIDADE_MODULO_RATEADA_CURSO'] = resumo_modulo_curso['CH_CAPACIDADE_MODULO_RATEADA_CURSO'].fillna(0)
resumo_modulo_curso['DELTA_CH_MODULO_CURSO'] = (
    resumo_modulo_curso['CH_CAPACIDADE_MODULO_RATEADA_CURSO'] - resumo_modulo_curso['DEMANDA_CH_MODULO_CURSO']
)

def _status_modulo_curso(row):
    if row['DEMANDA_CH_MODULO_CURSO'] > 0 and row['QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO'] == 0:
        return 'SEM TUTOR PARA O CURSO NO MODULO'
    if row['DELTA_CH_MODULO_CURSO'] < 0:
        return 'FALTA CH NO CURSO NO MODULO'
    if row['DELTA_CH_MODULO_CURSO'] > 0:
        return 'SOBRA CH NO CURSO NO MODULO'
    return 'CURSO EQUILIBRADO NO MODULO'

resumo_modulo_curso['STATUS_MODULO_CURSO'] = resumo_modulo_curso.apply(_status_modulo_curso, axis=1)

print(f"[Etapa 5A] Capacidade do módulo por curso calculada — {time.time()-t0:.1f}s")

## 5C. RASTREIO DE TUTORES COM CAPACIDADE NÃO UTILIZADA O rateio da Etapa 5A distribui 100% da capacidade de cada tutor entre os cursos em que ele está elegível, proporcional à demanda de cada curso no módulo — então a soma sempre "bate" no nível do tutor. A capacidade fica efetivamente OCIOSA quando o CURSO para o qual ela foi direcionada já tem mais capacidade do que demanda (SOBRA CH NO CURSO NO MODULO): a fatia do tutor naquele curso, proporcional à sobra do curso, é tempo alocado sem demanda real por trás. Esta etapa rastreia isso tutor a tutor.

In [ ]:
t0 = time.time()

rastreio_tutores = distrib_tutor_modulo.merge(
    resumo_modulo_curso[['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE',
                          'CH_CAPACIDADE_MODULO_RATEADA_CURSO',
                          'DELTA_CH_MODULO_CURSO', 'STATUS_MODULO_CURSO']],
    on=['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE'], how='left'
)

# Fatia do tutor na sobra do curso: proporcional à participação dele na
# capacidade total daquele curso naquele módulo.
rastreio_tutores['CH_NAO_UTILIZADA_TUTOR_CURSO'] = np.where(
    (rastreio_tutores['DELTA_CH_MODULO_CURSO'] > 0) & (rastreio_tutores['CH_CAPACIDADE_MODULO_RATEADA_CURSO'] > 0),
    rastreio_tutores['CH_CAPACIDADE_MODULO_RATEADA_TUTOR_CURSO']
    * (rastreio_tutores['DELTA_CH_MODULO_CURSO'] / rastreio_tutores['CH_CAPACIDADE_MODULO_RATEADA_CURSO']),
    0
)

rastreio_tutores = rastreio_tutores.rename(columns={
    'CH_SEMANAL': 'CH_SEMANAL_TUTOR',
    'NOME_CURSO_DEMANDA_MODULO': 'NOME_CURSO',
})
rastreio_tutores['CH_CAPACIDADE_TOTAL_MODULO_TUTOR'] = rastreio_tutores['CH_SEMANAL_TUTOR'] * SEMANAS_DO_MODULO
rastreio_tutores['PERCENTUAL_NAO_UTILIZADO_NO_CURSO'] = np.where(
    rastreio_tutores['CH_CAPACIDADE_MODULO_RATEADA_TUTOR_CURSO'] > 0,
    rastreio_tutores['CH_NAO_UTILIZADA_TUTOR_CURSO'] / rastreio_tutores['CH_CAPACIDADE_MODULO_RATEADA_TUTOR_CURSO'],
    0
)

# Resumo por tutor x módulo (SEM cidade — a capacidade é da pessoa, e ela
# pode estar espalhada por várias cidades/cursos no mesmo módulo). As colunas
# CIDADES_ELEGIVEIS e CURSOS_ELEGIVEIS listam onde e em quê esse tutor atua,
# para rastreabilidade — nomes de curso legíveis, não o código normalizado.
resumo_tutor_modulo = rastreio_tutores.groupby(['MATRICULA', 'MODULO'], dropna=False).agg(
    NOME_TUTOR=('NOME_TUTOR', _primeiro_texto),
    CH_SEMANAL_TUTOR=('CH_SEMANAL_TUTOR', 'max'),
    CH_CAPACIDADE_TOTAL_MODULO_TUTOR=('CH_CAPACIDADE_TOTAL_MODULO_TUTOR', 'max'),
    QTDE_CIDADES_ELEGIVEIS=('CIDADE_CHAVE', 'nunique'),
    CIDADES_ELEGIVEIS=('CIDADE_CHAVE', lambda s: ' | '.join(sorted(s.unique()))),
    QTDE_CURSOS_ELEGIVEIS=('CURSO_CHAVE', 'nunique'),
    CURSOS_ELEGIVEIS=('NOME_CURSO', lambda s: ' | '.join(sorted(s.dropna().unique()))),
    CH_NAO_UTILIZADA_TOTAL=('CH_NAO_UTILIZADA_TUTOR_CURSO', 'sum'),
).reset_index()
resumo_tutor_modulo['PERCENTUAL_NAO_UTILIZADO'] = np.where(
    resumo_tutor_modulo['CH_CAPACIDADE_TOTAL_MODULO_TUTOR'] > 0,
    resumo_tutor_modulo['CH_NAO_UTILIZADA_TOTAL'] / resumo_tutor_modulo['CH_CAPACIDADE_TOTAL_MODULO_TUTOR'],
    0
)
resumo_tutor_modulo = resumo_tutor_modulo.sort_values('CH_NAO_UTILIZADA_TOTAL', ascending=False)

COLUNAS_RASTREIO_DETALHE = [
    'CIDADE_CHAVE', 'MATRICULA', 'NOME_TUTOR', 'MODULO', 'CURSO_CHAVE', 'NOME_CURSO',
    'CH_SEMANAL_TUTOR', 'CH_CAPACIDADE_MODULO_RATEADA_TUTOR_CURSO', 'DEMANDA_CH_MODULO_CURSO',
    'CH_CAPACIDADE_MODULO_RATEADA_CURSO', 'DELTA_CH_MODULO_CURSO', 'STATUS_MODULO_CURSO',
    'CH_NAO_UTILIZADA_TUTOR_CURSO', 'PERCENTUAL_NAO_UTILIZADO_NO_CURSO',
]
rastreio_tutores_final = rastreio_tutores[COLUNAS_RASTREIO_DETALHE].rename(
    columns={'CIDADE_CHAVE': 'CIDADE', 'CURSO_CHAVE': 'CURSO'}
).sort_values(['CH_NAO_UTILIZADA_TUTOR_CURSO'], ascending=False).reset_index(drop=True)

print(f"[Etapa 5C] Rastreio de tutores calculado: {len(rastreio_tutores_final):,} linhas "
      f"(tutor x curso x módulo), {resumo_tutor_modulo['MATRICULA'].nunique():,} tutores únicos "
      f"— {time.time()-t0:.1f}s")
print(f"[Etapa 5C] Tutores com alguma CH não utilizada em algum módulo: "
      f"{(resumo_tutor_modulo['CH_NAO_UTILIZADA_TOTAL'] > 0.01).sum():,} de {len(resumo_tutor_modulo):,} "
      f"combinações tutor x módulo")

## 5B. CAPACIDADE POR PERÍODO (SEMANA) POR CURSO ELEGÍVEL

In [ ]:
t0 = time.time()

tutores_demanda = tutor_curso_cidade.merge(demanda_curso, on=['CIDADE_CHAVE', 'CURSO_CHAVE'], how='inner')

tutor_unico_periodo = tutores_demanda.groupby(['CIDADE_CHAVE', 'MATRICULA', 'MODULO', 'SEMANA_ANALISE'], dropna=False).agg(
    NOME_TUTOR=('NOME_TUTOR', _primeiro_texto),
    CH_SEMANAL_TUTOR=('CH_SEMANAL', lambda s: s.max() if s.notna().any() else 0),
).reset_index()

resumo_tutores_cidade = tutor_unico_periodo.groupby(['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE'], dropna=False).agg(
    QTDE_TUTORES_CIDADE_ELEGIVEIS=('MATRICULA', 'count'),
    CH_SEMANAL_TUTORES_CIDADE_ELEGIVEIS=('CH_SEMANAL_TUTOR', 'sum'),
).reset_index()

distrib_tutor = _rateio_vetorizado(
    tutores_demanda.copy(),
    ['MATRICULA', 'MODULO', 'SEMANA_ANALISE'],  # capacidade é da PESSOA, compartilhada
                                                 # entre todas as cidades onde é elegível
    'DEMANDA_CH_CURSO', 'CH_SEMANAL',
    'CH_RATEADA_TUTOR_CURSO'
)
# A rateio_vetorizado acima usa capacidade_tutor = ch_tutor * SEMANAS_DO_MODULO,
# mas para o nível semanal a capacidade deve ser SEMANAL (sem multiplicar por
# SEMANAS_DO_MODULO). Recalcula especificamente para este caso:
demanda_elegivel = distrib_tutor.groupby(['MATRICULA', 'MODULO', 'SEMANA_ANALISE'])['DEMANDA_CH_CURSO'].transform('sum')
ch_tutor_semanal = distrib_tutor.groupby(['MATRICULA', 'MODULO', 'SEMANA_ANALISE'])['CH_SEMANAL'].transform('max')
distrib_tutor['CH_RATEADA_TUTOR_CURSO'] = np.where(
    demanda_elegivel == 0, 0, ch_tutor_semanal * distrib_tutor['DEMANDA_CH_CURSO'] / demanda_elegivel
)

capacidade_curso = distrib_tutor.groupby(['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE', 'CURSO_CHAVE'], dropna=False).agg(
    QTDE_TUTORES_ELEGIVEIS_CURSO=('MATRICULA', 'nunique'),
    CH_CAPACIDADE_RATEADA_CURSO=('CH_RATEADA_TUTOR_CURSO', 'sum'),
    QTDE_TUTORES_CH_DIVERGENTE=('QTDE_VALORES_CH', lambda s: (s > 1).sum()),
).reset_index()

resumo_curso = demanda_curso.merge(
    capacidade_curso, on=['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE', 'CURSO_CHAVE'], how='left'
)
for c in ['QTDE_TUTORES_ELEGIVEIS_CURSO', 'CH_CAPACIDADE_RATEADA_CURSO', 'QTDE_TUTORES_CH_DIVERGENTE']:
    resumo_curso[c] = resumo_curso[c].fillna(0)
resumo_curso['DELTA_CH_CURSO'] = resumo_curso['CH_CAPACIDADE_RATEADA_CURSO'] - resumo_curso['DEMANDA_CH_CURSO']

def _status_curso(row):
    if row['DEMANDA_CH_CURSO'] > 0 and row['QTDE_TUTORES_ELEGIVEIS_CURSO'] == 0:
        return 'SEM TUTOR PARA O CURSO'
    if row['DELTA_CH_CURSO'] < 0:
        return 'FALTA CH NO CURSO'
    if row['DELTA_CH_CURSO'] > 0:
        return 'SOBRA CH NO CURSO'
    return 'CURSO EQUILIBRADO'

resumo_curso['STATUS_CURSO'] = resumo_curso.apply(_status_curso, axis=1)

print(f"[Etapa 5B] Capacidade por período/curso calculada — {time.time()-t0:.1f}s")

## 6. CAPACIDADE E DELTA ADITIVOS NA LINHA DA HIERARQUIA

In [ ]:
t0 = time.time()

detalhe = detalhe.merge(
    resumo_curso[['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE', 'CURSO_CHAVE',
                  'DEMANDA_CH_CURSO', 'QTDE_TUTORES_ELEGIVEIS_CURSO',
                  'CH_CAPACIDADE_RATEADA_CURSO', 'DELTA_CH_CURSO', 'STATUS_CURSO',
                  'QTDE_TUTORES_CH_DIVERGENTE']],
    on=['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE', 'CURSO_CHAVE'], how='left'
)

detalhe['CH_CAPACIDADE_AUX_RATEIO_CURSO_LINHA'] = np.where(
    (detalhe['DEMANDA_CH_CURSO'].isna()) | (detalhe['DEMANDA_CH_CURSO'] == 0),
    0,
    detalhe['CH_CAPACIDADE_RATEADA_CURSO'] * detalhe['DEMANDA_CH_AUX_RATEIO_CURSO'] / detalhe['DEMANDA_CH_CURSO']
)

detalhe = detalhe.merge(
    resumo_modulo_curso[['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE',
                          'DEMANDA_CH_MODULO_CURSO', 'QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO',
                          'CH_CAPACIDADE_MODULO_RATEADA_CURSO', 'DELTA_CH_MODULO_CURSO',
                          'STATUS_MODULO_CURSO']],
    on=['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE'], how='left'
)

detalhe['CH_CAPACIDADE_MODULO_AUX_RATEIO_CURSO_LINHA'] = np.where(
    (detalhe['DEMANDA_CH_MODULO_CURSO'].isna()) | (detalhe['DEMANDA_CH_MODULO_CURSO'] == 0),
    0,
    detalhe['CH_CAPACIDADE_MODULO_RATEADA_CURSO'] * detalhe['DEMANDA_CH_MODULO_AUX_RATEIO_CURSO'] / detalhe['DEMANDA_CH_MODULO_CURSO']
)

# Soma os rateios auxiliares por oferta e coloca o valor integral só na
# linha contabilizadora (evita duplicar capacidade ao somar a tabela toda).
capacidade_por_oferta = detalhe.groupby('ID_OFERTA', dropna=False).agg(
    CH_CAPACIDADE_RATEADA_OFERTA=('CH_CAPACIDADE_AUX_RATEIO_CURSO_LINHA', 'sum'),
    CH_CAPACIDADE_MODULO_RATEADA_OFERTA=('CH_CAPACIDADE_MODULO_AUX_RATEIO_CURSO_LINHA', 'sum'),
).reset_index()

detalhe = detalhe.merge(capacidade_por_oferta, on='ID_OFERTA', how='left')

detalhe['CH_CAPACIDADE_RATEADA_LINHA'] = np.where(
    detalhe['LINHA_CONTABILIZADORA_OFERTA'], detalhe['CH_CAPACIDADE_RATEADA_OFERTA'], 0
)
detalhe['DELTA_CH_RATEADO_LINHA'] = np.where(
    detalhe['LINHA_CONTABILIZADORA_OFERTA'],
    detalhe['CH_CAPACIDADE_RATEADA_OFERTA'] - detalhe['DEMANDA_CH_OFERTA'], 0
)
detalhe['CH_CAPACIDADE_MODULO_RATEADA_LINHA'] = np.where(
    detalhe['LINHA_CONTABILIZADORA_OFERTA'], detalhe['CH_CAPACIDADE_MODULO_RATEADA_OFERTA'], 0
)
detalhe['DELTA_CH_MODULO_LINHA'] = np.where(
    detalhe['LINHA_CONTABILIZADORA_OFERTA'],
    detalhe['CH_CAPACIDADE_MODULO_RATEADA_OFERTA'] - (detalhe['DEMANDA_CH_OFERTA'] * detalhe['FATOR_OCORRENCIAS_MODULO']),
    0
)

print(f"[Etapa 6] Delta calculado em nível de linha/curso/módulo — {time.time()-t0:.1f}s")

## RESUMO POR CIDADE E MÓDULO x CIDADE

In [ ]:
t0 = time.time()

resumo_cidade = detalhe.groupby(['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE'], dropna=False).agg(
    QTDE_POLOS_CIDADE=('UNIDADE_FISICA', 'nunique'),
    QTDE_CURSOS_CIDADE=('CURSO_CHAVE', 'nunique'),
    QTDE_DISCIPLINAS_CIDADE=('DISCIPLINA_NORMALIZADA', 'nunique'),
    QTDE_ALUNOS_MATRICULAS_CIDADE=('QTDE_ALUNOS_LINHA', 'sum'),
    QTDE_GRUPOS_REAIS_CIDADE=('QTDE_GRUPOS_CONTABILIZADOS', 'sum'),
    DEMANDA_CH_CIDADE=('DEMANDA_CH_RATEADA_LINHA', 'sum'),
    CH_CAPACIDADE_CIDADE=('CH_CAPACIDADE_RATEADA_LINHA', 'sum'),
).reset_index()

resumo_cidade = resumo_cidade.merge(
    resumo_tutores_cidade, on=['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE'], how='left'
)
for c in ['QTDE_TUTORES_CIDADE_ELEGIVEIS', 'CH_SEMANAL_TUTORES_CIDADE_ELEGIVEIS']:
    resumo_cidade[c] = resumo_cidade[c].fillna(0)

resumo_cidade['DELTA_CH_CIDADE'] = resumo_cidade['CH_CAPACIDADE_CIDADE'] - resumo_cidade['DEMANDA_CH_CIDADE']

def _status_cidade(row):
    if row['DEMANDA_CH_CIDADE'] > 0 and row['CH_CAPACIDADE_CIDADE'] == 0:
        return 'SEM TUTOR ELEGIVEL NA CIDADE'
    if row['DELTA_CH_CIDADE'] < 0:
        return 'FALTA CH NA CIDADE'
    if row['DELTA_CH_CIDADE'] > 0:
        return 'SOBRA CH NA CIDADE'
    return 'CIDADE EQUILIBRADA'

resumo_cidade['STATUS_CIDADE'] = resumo_cidade.apply(_status_cidade, axis=1)
resumo_cidade['DELTA_CH_CIDADE_FORMATADO'] = resumo_cidade['DELTA_CH_CIDADE'].apply(
    lambda v: f"+{v:.2f}".rstrip('0').rstrip('.') if v > 0 else f"{v:.2f}".rstrip('0').rstrip('.')
)

resumo_cidade_modulo = detalhe.groupby(['CIDADE_CHAVE', 'MODULO'], dropna=False).agg(
    QTDE_ALUNOS_MATRICULAS_MODULO_CIDADE=('QTDE_ALUNOS_MODULO_LINHA', 'sum'),
    QTDE_GRUPOS_FISICOS_MODULO_CIDADE=('QTDE_GRUPOS_FISICOS_MODULO_LINHA', 'sum'),
    DEMANDA_CH_MODULO_CIDADE=('DEMANDA_CH_MODULO_LINHA', 'sum'),
    CH_CAPACIDADE_MODULO_CIDADE=('CH_CAPACIDADE_MODULO_RATEADA_LINHA', 'sum'),
).reset_index()

resumo_cidade_modulo = resumo_cidade_modulo.merge(
    resumo_tutores_cidade_modulo, on=['CIDADE_CHAVE', 'MODULO'], how='left'
)
for c in ['QTDE_TUTORES_MODULO_CIDADE_ELEGIVEIS', 'CH_SEMANAL_TUTORES_MODULO_CIDADE_ELEGIVEIS',
          'CH_CAPACIDADE_TOTAL_MODULO_TUTORES_ELEGIVEIS']:
    resumo_cidade_modulo[c] = resumo_cidade_modulo[c].fillna(0)

resumo_cidade_modulo['DELTA_CH_MODULO_CIDADE'] = (
    resumo_cidade_modulo['CH_CAPACIDADE_MODULO_CIDADE'] - resumo_cidade_modulo['DEMANDA_CH_MODULO_CIDADE']
)

def _status_modulo_cidade(row):
    if row['DEMANDA_CH_MODULO_CIDADE'] > 0 and row['CH_CAPACIDADE_MODULO_CIDADE'] == 0:
        return 'SEM TUTOR ELEGIVEL NA CIDADE NO MODULO'
    if row['DELTA_CH_MODULO_CIDADE'] < 0:
        return 'FALTA CH NA CIDADE NO MODULO'
    if row['DELTA_CH_MODULO_CIDADE'] > 0:
        return 'SOBRA CH NA CIDADE NO MODULO'
    return 'CIDADE EQUILIBRADA NO MODULO'

resumo_cidade_modulo['STATUS_MODULO_CIDADE'] = resumo_cidade_modulo.apply(_status_modulo_cidade, axis=1)
resumo_cidade_modulo['DELTA_CH_MODULO_CIDADE_FORMATADO'] = resumo_cidade_modulo['DELTA_CH_MODULO_CIDADE'].apply(
    lambda v: f"+{v:.2f}".rstrip('0').rstrip('.') if v > 0 else f"{v:.2f}".rstrip('0').rstrip('.')
)

detalhe = detalhe.merge(resumo_cidade, on=['CIDADE_CHAVE', 'MODULO', 'SEMANA_ANALISE'], how='left', suffixes=('', '_CIDADE_NAO_SOMAR'))
detalhe = detalhe.merge(resumo_cidade_modulo, on=['CIDADE_CHAVE', 'MODULO'], how='left', suffixes=('', '_MODCID_NAO_SOMAR'))

print(f"[Etapa 7] Resumos de cidade e módulo x cidade calculados — {time.time()-t0:.1f}s")
print(f"\nTempo total de execução: {time.time()-t_inicio:.1f}s")

## 7B. DEMANDA E ALOCAÇÃO POR POLO (UNIDADE FÍSICA) — aba `RESUMO_POLO_MODULO`

Uma linha por **polo × módulo × curso**, com a informação correspondente a cada curso naquele polo: alunos, grupos, demanda, capacidade alocada, delta e status. As colunas sufixadas `_POLO_NAO_SOMAR` repetem o total do polo em cada linha, para contexto — não devem ser somadas entre linhas.

A cidade é o nível **oficial** do delta: o tutor atende todos os polos do município, então a capacidade é um pool municipal, não uma lotação por unidade. Esta etapa distribui esse pool entre os polos proporcionalmente à demanda de cada um; como os pesos somam 1, a soma dos polos reconcilia exatamente com a cidade.

Vale registrar a consequência: sendo o rateio proporcional à demanda, o *percentual* de cobertura é idêntico em todos os polos de uma cidade para um mesmo curso. A aba mostra onde o déficit se materializa em horas absolutas — priorização e logística —, não um diagnóstico independente. Um polo só acende por conta própria quando há demanda nele e a cidade inteira não tem tutor para o curso.

In [ ]:
t0 = time.time()

# A CH de preparação ("mentoria") existe no `detalhe` em duas formas: a da
# LINHA CONTABILIZADORA (preparação inteira da oferta, atribuída a um único
# curso) e a AUX_RATEIO_CURSO (rateada entre os cursos pelo peso de alunos).
# No grão de curso é obrigatório usar a rateada — a outra daria 100% da
# preparação ao curso dono da linha e zero ao parceiro de sinergia. O pipeline
# já tinha a versão de módulo para a demanda, mas não para a preparação;
# criamos aqui, pela mesma fórmula.
detalhe['CH_MENTORIA_MODULO_AUX_RATEIO_CURSO'] = (
    detalhe['CH_PREPARACAO_AUX_RATEIO_CURSO'] * detalhe['FATOR_OCORRENCIAS_MODULO']
)

demanda_modulo_curso_polo = detalhe.groupby(
    ['CIDADE_CHAVE', 'UNIDADE_FISICA', 'MODULO', 'CURSO_CHAVE'], dropna=False
).agg(
    NM_CIDADE_POLO=('NM_CIDADE', _primeiro_texto),
    UF_POLO=('CD_UF', _primeiro_texto),
    NM_UNIDADE_POLO=('NM_UNIDADE', _primeiro_texto),
    NOME_CURSO_POLO=('NOME_CURSO', _primeiro_texto),
    CLUSTER_POLO=('CLUSTER', _primeiro_texto),
    QTDE_ALUNOS_MODULO_POLO_CURSO=('QTDE_ALUNOS_MODULO_LINHA', 'sum'),
    QTDE_GRUPOS_MODULO_POLO_CURSO=('QTDE_GRUPOS_FISICOS_MODULO_LINHA', 'sum'),
    QTDE_DISCIPLINAS_POLO_CURSO=('DISCIPLINA_NORMALIZADA', 'nunique'),
    CAPACIDADE_LPI_MIN=('CAPACIDADE_LPI', 'min'),
    CAPACIDADE_LPI_MAX=('CAPACIDADE_LPI', 'max'),
    QTD_PRATICAS_MODULO_MIN=('QTD_PRATICAS_MODULO', 'min'),
    QTD_PRATICAS_MODULO_MAX=('QTD_PRATICAS_MODULO', 'max'),
    TEM_SINERGIA_POLO_CURSO=('SINERGIA_VALIDA_LINHA', 'any'),
    CH_MENTORIA_MODULO_CURSO_POLO=('CH_MENTORIA_MODULO_AUX_RATEIO_CURSO', 'sum'),
    DEMANDA_CH_MODULO_CURSO_POLO=('DEMANDA_CH_MODULO_AUX_RATEIO_CURSO', 'sum'),
).reset_index()

demanda_modulo_curso_polo['TEM_SINERGIA_POLO_CURSO'] = np.where(
    demanda_modulo_curso_polo['TEM_SINERGIA_POLO_CURSO'], 'SIM', 'NAO'
)

# O pool de capacidade é MUNICIPAL. Cada polo recebe a fatia proporcional à sua
# participação na demanda daquele curso na cidade — os pesos somam 1, então a
# soma dos polos reconcilia exatamente com o total da cidade.
resumo_polo_modulo_curso = demanda_modulo_curso_polo.merge(
    resumo_modulo_curso[['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE',
                         'DEMANDA_CH_MODULO_CURSO', 'CH_CAPACIDADE_MODULO_RATEADA_CURSO',
                         'QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO']],
    on=['CIDADE_CHAVE', 'MODULO', 'CURSO_CHAVE'], how='left'
)

resumo_polo_modulo_curso['PESO_DEMANDA_POLO_NA_CIDADE'] = np.where(
    resumo_polo_modulo_curso['DEMANDA_CH_MODULO_CURSO'].fillna(0) == 0, 0,
    resumo_polo_modulo_curso['DEMANDA_CH_MODULO_CURSO_POLO']
    / resumo_polo_modulo_curso['DEMANDA_CH_MODULO_CURSO']
)
resumo_polo_modulo_curso['CH_CAPACIDADE_MODULO_ALOCADA_CURSO_POLO'] = (
    resumo_polo_modulo_curso['CH_CAPACIDADE_MODULO_RATEADA_CURSO'].fillna(0)
    * resumo_polo_modulo_curso['PESO_DEMANDA_POLO_NA_CIDADE']
)
resumo_polo_modulo_curso['DELTA_CH_MODULO_CURSO_POLO'] = (
    resumo_polo_modulo_curso['CH_CAPACIDADE_MODULO_ALOCADA_CURSO_POLO']
    - resumo_polo_modulo_curso['DEMANDA_CH_MODULO_CURSO_POLO']
)
resumo_polo_modulo_curso['QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO'] = (
    resumo_polo_modulo_curso['QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO'].fillna(0)
)

def _status_curso_polo(row):
    if (row['DEMANDA_CH_MODULO_CURSO_POLO'] > 0
            and row['QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO'] == 0):
        return 'SEM TUTOR NA CIDADE PARA O CURSO'
    if row['DELTA_CH_MODULO_CURSO_POLO'] < -0.01:
        return 'FALTA CH NO CURSO NO POLO'
    if row['DELTA_CH_MODULO_CURSO_POLO'] > 0.01:
        return 'SOBRA CH NO CURSO NO POLO'
    return 'CURSO EQUILIBRADO NO POLO'

resumo_polo_modulo_curso['STATUS_MODULO_CURSO_POLO'] = (
    resumo_polo_modulo_curso.apply(_status_curso_polo, axis=1)
)

# Totais do polo, repetidos em cada linha apenas como contexto.
totais_polo = resumo_polo_modulo_curso.groupby(
    ['UNIDADE_FISICA', 'MODULO'], dropna=False
).agg(
    QTDE_CURSOS_POLO=('CURSO_CHAVE', 'nunique'),
    QTDE_ALUNOS_MODULO_POLO=('QTDE_ALUNOS_MODULO_POLO_CURSO', 'sum'),
    QTDE_GRUPOS_MODULO_POLO=('QTDE_GRUPOS_MODULO_POLO_CURSO', 'sum'),
    CURSOS_COM_SINERGIA_POLO=('TEM_SINERGIA_POLO_CURSO', lambda s: (s == 'SIM').sum()),
    CH_MENTORIA_MODULO_POLO=('CH_MENTORIA_MODULO_CURSO_POLO', 'sum'),
    DEMANDA_CH_MODULO_POLO=('DEMANDA_CH_MODULO_CURSO_POLO', 'sum'),
    CH_CAPACIDADE_MODULO_POLO=('CH_CAPACIDADE_MODULO_ALOCADA_CURSO_POLO', 'sum'),
    CURSOS_SEM_TUTOR_NA_CIDADE=('STATUS_MODULO_CURSO_POLO',
                                lambda s: (s == 'SEM TUTOR NA CIDADE PARA O CURSO').sum()),
).reset_index()
totais_polo['DELTA_CH_MODULO_POLO'] = (
    totais_polo['CH_CAPACIDADE_MODULO_POLO'] - totais_polo['DEMANDA_CH_MODULO_POLO']
)

def _status_polo(row):
    if row['DEMANDA_CH_MODULO_POLO'] > 0 and row['CH_CAPACIDADE_MODULO_POLO'] == 0:
        return 'SEM TUTOR NA CIDADE'
    if row['DELTA_CH_MODULO_POLO'] < -0.01:
        return 'FALTA CH NO POLO'
    if row['DELTA_CH_MODULO_POLO'] > 0.01:
        return 'SOBRA CH NO POLO'
    return 'POLO EQUILIBRADO'

totais_polo['STATUS_MODULO_POLO'] = totais_polo.apply(_status_polo, axis=1)
totais_polo = totais_polo.rename(columns={
    'QTDE_CURSOS_POLO': 'QTDE_CURSOS_POLO_NAO_SOMAR',
    'QTDE_ALUNOS_MODULO_POLO': 'QTDE_ALUNOS_MODULO_POLO_NAO_SOMAR',
    'QTDE_GRUPOS_MODULO_POLO': 'QTDE_GRUPOS_MODULO_POLO_NAO_SOMAR',
    'CURSOS_COM_SINERGIA_POLO': 'CURSOS_COM_SINERGIA_POLO_NAO_SOMAR',
    'CH_MENTORIA_MODULO_POLO': 'CH_MENTORIA_MODULO_POLO_NAO_SOMAR',
    'DEMANDA_CH_MODULO_POLO': 'DEMANDA_CH_MODULO_POLO_NAO_SOMAR',
    'CH_CAPACIDADE_MODULO_POLO': 'CH_CAPACIDADE_MODULO_POLO_NAO_SOMAR',
    'DELTA_CH_MODULO_POLO': 'DELTA_CH_MODULO_POLO_NAO_SOMAR',
    'CURSOS_SEM_TUTOR_NA_CIDADE': 'CURSOS_SEM_TUTOR_NA_CIDADE_NAO_SOMAR',
})

resumo_polo_modulo = resumo_polo_modulo_curso.merge(
    totais_polo, on=['UNIDADE_FISICA', 'MODULO'], how='left'
)

COLUNAS_POLO_FINAIS = [
    'NM_CIDADE_POLO', 'UF_POLO', 'UNIDADE_FISICA', 'NM_UNIDADE_POLO', 'MODULO',
    'CURSO_CHAVE', 'NOME_CURSO_POLO', 'CLUSTER_POLO',
    'QTDE_DISCIPLINAS_POLO_CURSO', 'TEM_SINERGIA_POLO_CURSO',
    'CAPACIDADE_LPI_MIN', 'CAPACIDADE_LPI_MAX',
    'QTD_PRATICAS_MODULO_MIN', 'QTD_PRATICAS_MODULO_MAX',
    'QTDE_ALUNOS_MODULO_POLO_CURSO', 'QTDE_GRUPOS_MODULO_POLO_CURSO',
    'DEMANDA_CH_MODULO_CURSO_POLO', 'CH_MENTORIA_MODULO_CURSO_POLO',
    'PESO_DEMANDA_POLO_NA_CIDADE',
    'QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO',
    'CH_CAPACIDADE_MODULO_ALOCADA_CURSO_POLO', 'DELTA_CH_MODULO_CURSO_POLO',
    'STATUS_MODULO_CURSO_POLO',
    'QTDE_CURSOS_POLO_NAO_SOMAR', 'CURSOS_COM_SINERGIA_POLO_NAO_SOMAR',
    'QTDE_ALUNOS_MODULO_POLO_NAO_SOMAR',
    'QTDE_GRUPOS_MODULO_POLO_NAO_SOMAR', 'DEMANDA_CH_MODULO_POLO_NAO_SOMAR',
    'CH_MENTORIA_MODULO_POLO_NAO_SOMAR',
    'CH_CAPACIDADE_MODULO_POLO_NAO_SOMAR', 'DELTA_CH_MODULO_POLO_NAO_SOMAR',
    'CURSOS_SEM_TUTOR_NA_CIDADE_NAO_SOMAR', 'STATUS_MODULO_POLO',
]
resumo_polo_modulo = resumo_polo_modulo[
    [c for c in COLUNAS_POLO_FINAIS if c in resumo_polo_modulo.columns]
].sort_values(['NM_CIDADE_POLO', 'NM_UNIDADE_POLO', 'MODULO', 'NOME_CURSO_POLO'])

cap_cidade = resumo_modulo_curso['CH_CAPACIDADE_MODULO_RATEADA_CURSO'].sum()
cap_polo = resumo_polo_modulo['CH_CAPACIDADE_MODULO_ALOCADA_CURSO_POLO'].sum()
print(f"[Etapa 7B] Capacidade distribuída entre polos: {cap_polo:,.0f}h de "
      f"{cap_cidade:,.0f}h da cidade ({cap_polo/max(cap_cidade,1):.2%} — deve fechar 100%)")
print(f"[Etapa 7B] Linhas polo x módulo x curso: {len(resumo_polo_modulo):,} | "
      f"polos: {resumo_polo_modulo['UNIDADE_FISICA'].nunique():,} — {time.time()-t0:.1f}s")


## STATUS DE QUALIDADE E EXPORTAÇÃO FINAL

In [ ]:
t0 = time.time()

def _status_qualidade(row):
    alertas = []
    if 'DESCARTADA' in str(row['MOTIVO_REGRA_SINERGIA']):
        alertas.append(row['MOTIVO_REGRA_SINERGIA'])
    if row['SEMANA_ANALISE'] == 'NSA':
        alertas.append('PERIODO NSA')
    if row['QTDE_TUTORES_ELEGIVEIS_CURSO'] == 0:
        alertas.append('SEM TUTOR PARA O CURSO')
    if row['QTDE_TUTORES_CH_DIVERGENTE'] > 0:
        alertas.append('CH DO TUTOR DIVERGENTE')
    return ' | '.join(alertas) if alertas else 'OK'

detalhe['STATUS_QUALIDADE'] = detalhe.apply(_status_qualidade, axis=1)

# Renomeia para deixar explícito que essas colunas NÃO devem ser somadas
# entre linhas de disciplina/módulo diferentes (repetem o mesmo valor por
# cidade ou módulo x cidade) — mesma convenção usada no Power Query.
RENOMEAR_NAO_SOMAR = {
    'QTDE_POLOS_CIDADE': 'QTDE_POLOS_CIDADE_NAO_SOMAR',
    'QTDE_CURSOS_CIDADE': 'QTDE_CURSOS_CIDADE_NAO_SOMAR',
    'QTDE_ALUNOS_MATRICULAS_CIDADE': 'QTDE_ALUNOS_MATRICULAS_CIDADE_NAO_SOMAR',
    'QTDE_GRUPOS_REAIS_CIDADE': 'QTDE_GRUPOS_REAIS_CIDADE_NAO_SOMAR',
    'DEMANDA_CH_CIDADE': 'DEMANDA_CH_CIDADE_NAO_SOMAR',
    'CH_CAPACIDADE_CIDADE': 'CH_CAPACIDADE_CIDADE_NAO_SOMAR',
    'QTDE_TUTORES_CIDADE_ELEGIVEIS': 'QTDE_TUTORES_CIDADE_ELEGIVEIS_NAO_SOMAR',
    'DELTA_CH_CIDADE': 'DELTA_CH_CIDADE_NAO_SOMAR',
    'QTDE_ALUNOS_MATRICULAS_MODULO_CIDADE': 'QTDE_ALUNOS_MATRICULAS_MODULO_CIDADE_NAO_SOMAR',
    'QTDE_GRUPOS_FISICOS_MODULO_CIDADE': 'QTDE_GRUPOS_FISICOS_MODULO_CIDADE_NAO_SOMAR',
    'DEMANDA_CH_MODULO_CIDADE': 'DEMANDA_CH_MODULO_CIDADE_NAO_SOMAR',
    'CH_CAPACIDADE_MODULO_CIDADE': 'CH_CAPACIDADE_MODULO_CIDADE_NAO_SOMAR',
    'QTDE_TUTORES_MODULO_CIDADE_ELEGIVEIS': 'QTDE_TUTORES_MODULO_CIDADE_ELEGIVEIS_NAO_SOMAR',
    'CH_CAPACIDADE_TOTAL_MODULO_TUTORES_ELEGIVEIS': 'CH_CAPACIDADE_TOTAL_MODULO_TUTORES_ELEGIVEIS_NAO_SOMAR',
    'DELTA_CH_MODULO_CIDADE': 'DELTA_CH_MODULO_CIDADE_NAO_SOMAR',
}
detalhe = detalhe.rename(columns=RENOMEAR_NAO_SOMAR)

# Colunas de diagnóstico: cidade -> polo -> curso -> disciplina -> delta.
# Descarta os rateios auxiliares internos (não têm valor analítico sozinhos).
COLUNAS_FINAIS = [
    # Hierarquia
    'NM_CIDADE', 'CD_UF', 'UNIDADE_FISICA', 'NM_UNIDADE', 'CLUSTER',
    'CURSO_AGRUPADO', 'NOME_CURSO', 'NOME_DISCIPLINA', 'MODULO', 'SEMANA_ANALISE',
    # Contexto da oferta/sinergia
    'ID_OFERTA', 'LINHA_CONTABILIZADORA_OFERTA', 'LINHA_CONTABILIZADORA_GRUPO_MODULO',
    'LINHA_CONTABILIZADORA_ALUNOS_MODULO', 'STATUS_SINERGIA', 'SINERGIA_APLICADA',
    'QTDE_CURSOS_PRESENTES_OFERTA', 'CAPACIDADE_LPI',
    'QTD_PRATICAS_MODULO', 'FATOR_OCORRENCIAS_MODULO',
    'QTDE_GRUPOS_CURSO_ISOLADO', 'GRUPOS_REDUZIDOS_SINERGIA',
    # Nível linha/semana (somam)
    'QTDE_ALUNOS_LINHA', 'QTDE_ALUNOS_MODULO_LINHA', 'QTDE_GRUPOS_RATEADOS_LINHA',
    'QTDE_GRUPOS_FISICOS_MODULO_LINHA', 'QTDE_GRUPOS_ATENDIMENTOS_MODULO_LINHA',
    'DEMANDA_CH_RATEADA_LINHA', 'DEMANDA_CH_MODULO_LINHA',
    'CH_PREPARACAO_RATEADA_LINHA', 'CH_PREPARACAO_MODULO_LINHA',
    'CH_CAPACIDADE_RATEADA_LINHA', 'CH_CAPACIDADE_MODULO_RATEADA_LINHA',
    'DELTA_CH_RATEADO_LINHA', 'DELTA_CH_MODULO_LINHA',
    # Diagnóstico por curso
    'DEMANDA_CH_CURSO', 'QTDE_TUTORES_ELEGIVEIS_CURSO', 'CH_CAPACIDADE_RATEADA_CURSO',
    'DELTA_CH_CURSO', 'STATUS_CURSO', 'DEMANDA_CH_MODULO_CURSO',
    'QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO', 'CH_CAPACIDADE_MODULO_RATEADA_CURSO',
    'DELTA_CH_MODULO_CURSO', 'STATUS_MODULO_CURSO',
    # Diagnóstico por cidade
    'QTDE_POLOS_CIDADE_NAO_SOMAR', 'QTDE_CURSOS_CIDADE_NAO_SOMAR',
    'QTDE_ALUNOS_MATRICULAS_CIDADE_NAO_SOMAR', 'QTDE_GRUPOS_REAIS_CIDADE_NAO_SOMAR',
    'DEMANDA_CH_CIDADE_NAO_SOMAR', 'CH_CAPACIDADE_CIDADE_NAO_SOMAR',
    'QTDE_TUTORES_CIDADE_ELEGIVEIS_NAO_SOMAR', 'DELTA_CH_CIDADE_NAO_SOMAR', 'STATUS_CIDADE',
    # Diagnóstico módulo x cidade
    'QTDE_ALUNOS_MATRICULAS_MODULO_CIDADE_NAO_SOMAR', 'QTDE_GRUPOS_FISICOS_MODULO_CIDADE_NAO_SOMAR',
    'DEMANDA_CH_MODULO_CIDADE_NAO_SOMAR', 'CH_CAPACIDADE_MODULO_CIDADE_NAO_SOMAR',
    'QTDE_TUTORES_MODULO_CIDADE_ELEGIVEIS_NAO_SOMAR',
    'CH_CAPACIDADE_TOTAL_MODULO_TUTORES_ELEGIVEIS_NAO_SOMAR',
    'DELTA_CH_MODULO_CIDADE_NAO_SOMAR', 'STATUS_MODULO_CIDADE',
    # Qualidade
    'STATUS_QUALIDADE',
]

resultado = detalhe[[c for c in COLUNAS_FINAIS if c in detalhe.columns]].copy()
resultado = resultado.sort_values(
    ['NM_CIDADE', 'UNIDADE_FISICA', 'NOME_CURSO', 'NOME_DISCIPLINA', 'MODULO', 'SEMANA_ANALISE']
).reset_index(drop=True)

## ETAPA 9. VISÃO SEMANAL MÉDIA (módulo ÷ 10)

Duas abas derivadas: `ANALISE_DISCIPLINA_SEMANA` (a partir de `ANALISE_CIDADE_CH`) e `RESUMO_POLO_SEMANA` (a partir de `RESUMO_POLO_MODULO`). Cada coluna de hora `_MODULO` é dividida por `SEMANAS_DO_MODULO` e renomeada para `_SEMANA_MEDIA`.

**Por que `_SEMANA_MEDIA` e não `_SEMANA`.** O pipeline já produz uma visão semanal *real*: as colunas sem `_MODULO` (`DEMANDA_CH_RATEADA_LINHA`, `DELTA_CH_CURSO`, etc.) são calculadas por `SEMANA_ANALISE` e respeitam o campo `SEMANA_DA_PRATICA` — uma prática que ocorre nas semanas 3, 6 e 9 gera carga só nessas três. A divisão por 10 espalha a mesma carga uniformemente pelas dez semanas, o que **não** descreve nenhuma semana do calendário.

As duas visões servem a perguntas diferentes:

| Pergunta | Use |
|---|---|
| "Minha CH semanal contratada cobre a carga média?" | `_SEMANA_MEDIA` (esta etapa) |
| "Qual semana do módulo vai estourar?" | Colunas por `SEMANA_ANALISE` (já existentes) |

**Contagens não são divididas.** Alunos, grupos, disciplinas e tutores permanecem no valor do módulo — meio grupo por semana não significa nada. Só horas são convertidas.

Por construção, `soma(_SEMANA_MEDIA) × 10 = soma(_MODULO)` para toda coluna convertida. A célula de validação confere isso com `assert`, coluna por coluna.

In [ ]:
t0 = time.time()

def gerar_visao_semanal(df, colunas_identificacao, colunas_hora_modulo, semanas=SEMANAS_DO_MODULO):
    """Divide as colunas de hora do módulo por `semanas` e renomeia _MODULO -> _SEMANA_MEDIA.

    Colunas de contagem (alunos, grupos, disciplinas) são mantidas no valor do
    módulo de propósito: dividir uma contagem por 10 produz fração sem sentido
    operacional. Só horas são convertidas.
    """
    presentes_id = [c for c in colunas_identificacao if c in df.columns]
    presentes_hora = [c for c in colunas_hora_modulo if c in df.columns]
    out = df[presentes_id + presentes_hora].copy()

    renomear = {}
    for c in presentes_hora:
        out[c] = out[c] / semanas
        renomear[c] = (c.replace('_MODULO', '_SEMANA_MEDIA') if '_MODULO' in c
                       else f'{c}_SEMANA_MEDIA')
    return out.rename(columns=renomear), list(renomear.values())


# --- Aba 1: nível disciplina, derivada de ANALISE_CIDADE_CH ---
ID_DISCIPLINA_SEMANA = [
    'NM_CIDADE', 'CD_UF', 'UNIDADE_FISICA', 'NM_UNIDADE', 'MODULO',
    'COD_CURSO', 'NOME_CURSO', 'CLUSTER', 'COD_DISCIPLINA', 'NOME_DISCIPLINA',
    'CAPACIDADE_LPI', 'QTD_PRATICAS_MODULO', 'FATOR_OCORRENCIAS_MODULO',
    'STATUS_SINERGIA', 'SINERGIA_APLICADA', 'GRUPOS_REDUZIDOS_SINERGIA',
    'QTDE_ALUNOS_MODULO_LINHA',
    'QTDE_GRUPOS_FISICOS_MODULO_LINHA', 'STATUS_MODULO_CURSO',
]
HORA_DISCIPLINA_MODULO = [
    'DEMANDA_CH_MODULO_LINHA', 'CH_PREPARACAO_MODULO_LINHA',
    'CH_CAPACIDADE_MODULO_RATEADA_LINHA', 'DELTA_CH_MODULO_LINHA',
]
analise_disciplina_semana, COLS_SEMANA_DISCIPLINA = gerar_visao_semanal(
    resultado, ID_DISCIPLINA_SEMANA, HORA_DISCIPLINA_MODULO
)

# --- Aba 2: nível polo x curso, derivada de RESUMO_POLO_MODULO ---
ID_POLO_SEMANA = [
    'NM_CIDADE_POLO', 'UF_POLO', 'UNIDADE_FISICA', 'NM_UNIDADE_POLO', 'MODULO',
    'CURSO_CHAVE', 'NOME_CURSO_POLO', 'CLUSTER_POLO', 'QTDE_DISCIPLINAS_POLO_CURSO',
    'TEM_SINERGIA_POLO_CURSO', 'CAPACIDADE_LPI_MIN', 'CAPACIDADE_LPI_MAX',
    'QTD_PRATICAS_MODULO_MIN', 'QTD_PRATICAS_MODULO_MAX',
    'QTDE_ALUNOS_MODULO_POLO_CURSO', 'QTDE_GRUPOS_MODULO_POLO_CURSO',
    'QTDE_TUTORES_ELEGIVEIS_MODULO_CURSO', 'PESO_DEMANDA_POLO_NA_CIDADE',
    'STATUS_MODULO_CURSO_POLO', 'STATUS_MODULO_POLO',
]
HORA_POLO_MODULO = [
    'DEMANDA_CH_MODULO_CURSO_POLO', 'CH_MENTORIA_MODULO_CURSO_POLO',
    'CH_CAPACIDADE_MODULO_ALOCADA_CURSO_POLO', 'DELTA_CH_MODULO_CURSO_POLO',
    'DEMANDA_CH_MODULO_POLO_NAO_SOMAR', 'CH_MENTORIA_MODULO_POLO_NAO_SOMAR',
    'CH_CAPACIDADE_MODULO_POLO_NAO_SOMAR', 'DELTA_CH_MODULO_POLO_NAO_SOMAR',
]
resumo_polo_semana, COLS_SEMANA_POLO = gerar_visao_semanal(
    resumo_polo_modulo, ID_POLO_SEMANA, HORA_POLO_MODULO
)

print(f"[Etapa 9] ANALISE_DISCIPLINA_SEMANA: {len(analise_disciplina_semana):,} linhas x "
      f"{len(analise_disciplina_semana.columns)} colunas")
print(f"[Etapa 9] RESUMO_POLO_SEMANA: {len(resumo_polo_semana):,} linhas x "
      f"{len(resumo_polo_semana.columns)} colunas")
print(f"[Etapa 9] Concluída — {time.time()-t0:.1f}s")


## COLUNAS LEGÍVEIS "Xh Ymin" — aplicadas a toda métrica de carga horária do resultado final e das duas abas de rastreio de tutores. As colunas numéricas originais permanecem intactas ao lado, para uso em fórmulas, somas e tabelas dinâmicas.

In [ ]:
COLUNAS_HORA_RESULTADO = [
    'DEMANDA_CH_RATEADA_LINHA', 'DEMANDA_CH_MODULO_LINHA',
    'CH_PREPARACAO_RATEADA_LINHA', 'CH_PREPARACAO_MODULO_LINHA',
    'CH_CAPACIDADE_RATEADA_LINHA', 'CH_CAPACIDADE_MODULO_RATEADA_LINHA',
    'DELTA_CH_RATEADO_LINHA', 'DELTA_CH_MODULO_LINHA',
    'DEMANDA_CH_CURSO', 'CH_CAPACIDADE_RATEADA_CURSO', 'DELTA_CH_CURSO',
    'DEMANDA_CH_MODULO_CURSO', 'CH_CAPACIDADE_MODULO_RATEADA_CURSO', 'DELTA_CH_MODULO_CURSO',
    'DEMANDA_CH_CIDADE_NAO_SOMAR', 'CH_CAPACIDADE_CIDADE_NAO_SOMAR', 'DELTA_CH_CIDADE_NAO_SOMAR',
    'DEMANDA_CH_MODULO_CIDADE_NAO_SOMAR', 'CH_CAPACIDADE_MODULO_CIDADE_NAO_SOMAR',
    'CH_CAPACIDADE_TOTAL_MODULO_TUTORES_ELEGIVEIS_NAO_SOMAR', 'DELTA_CH_MODULO_CIDADE_NAO_SOMAR',
]
resultado = adicionar_colunas_legiveis(resultado, COLUNAS_HORA_RESULTADO)

COLUNAS_HORA_RASTREIO_RESUMO = [
    'CH_SEMANAL_TUTOR', 'CH_CAPACIDADE_TOTAL_MODULO_TUTOR', 'CH_NAO_UTILIZADA_TOTAL',
]
resumo_tutor_modulo = adicionar_colunas_legiveis(resumo_tutor_modulo, COLUNAS_HORA_RASTREIO_RESUMO)

COLUNAS_HORA_RASTREIO_DETALHE = [
    'CH_SEMANAL_TUTOR', 'CH_CAPACIDADE_MODULO_RATEADA_TUTOR_CURSO', 'DEMANDA_CH_MODULO_CURSO',
    'CH_CAPACIDADE_MODULO_RATEADA_CURSO', 'DELTA_CH_MODULO_CURSO', 'CH_NAO_UTILIZADA_TUTOR_CURSO',
]
rastreio_tutores_final = adicionar_colunas_legiveis(rastreio_tutores_final, COLUNAS_HORA_RASTREIO_DETALHE)

COLUNAS_HORA_POLO = [
    'DEMANDA_CH_MODULO_CURSO_POLO', 'CH_CAPACIDADE_MODULO_ALOCADA_CURSO_POLO',
    'DELTA_CH_MODULO_CURSO_POLO', 'CH_MENTORIA_MODULO_CURSO_POLO',
    'CH_MENTORIA_MODULO_POLO_NAO_SOMAR', 'DEMANDA_CH_MODULO_POLO_NAO_SOMAR',
    'CH_CAPACIDADE_MODULO_POLO_NAO_SOMAR', 'DELTA_CH_MODULO_POLO_NAO_SOMAR',
]
resumo_polo_modulo = adicionar_colunas_legiveis(resumo_polo_modulo, COLUNAS_HORA_POLO)

analise_disciplina_semana = adicionar_colunas_legiveis(
    analise_disciplina_semana, COLS_SEMANA_DISCIPLINA)
resumo_polo_semana = adicionar_colunas_legiveis(resumo_polo_semana, COLS_SEMANA_POLO)

os.makedirs(PASTA_SAIDA, exist_ok=True)
CAMINHO_SAIDA = f'{PASTA_SAIDA}/ANALISE_CIDADE_CH{SUFIXO_SAIDA}.xlsx'

with pd.ExcelWriter(CAMINHO_SAIDA, engine='openpyxl') as writer:
    resultado.to_excel(writer, index=False, sheet_name='ANALISE_CIDADE_CH')
    resumo_polo_modulo.to_excel(writer, index=False, sheet_name='RESUMO_POLO_MODULO')
    analise_disciplina_semana.to_excel(writer, index=False, sheet_name='ANALISE_DISCIPLINA_SEMANA')
    resumo_polo_semana.to_excel(writer, index=False, sheet_name='RESUMO_POLO_SEMANA')
    resumo_tutor_modulo.to_excel(writer, index=False, sheet_name='RASTREIO_TUTORES_RESUMO')
    rastreio_tutores_final.to_excel(writer, index=False, sheet_name='RASTREIO_TUTORES_DETALHE')

print(f"[Exportação] {len(resultado):,} linhas x {len(resultado.columns)} colunas "
      f"salvas em {CAMINHO_SAIDA} (aba ANALISE_CIDADE_CH)")
print(f"[Exportação] Rastreio de tutores salvo nas abas RASTREIO_TUTORES_RESUMO "
      f"({len(resumo_tutor_modulo):,} linhas) e RASTREIO_TUTORES_DETALHE "
      f"({len(rastreio_tutores_final):,} linhas) — {time.time()-t0:.1f}s")
print(f"[Exportação] Aba RESUMO_POLO_MODULO: {len(resumo_polo_modulo):,} linhas "
      f"(polo x módulo x curso)")
print(f"[Exportação] Abas semanais: ANALISE_DISCIPLINA_SEMANA "
      f"({len(analise_disciplina_semana):,} linhas) e RESUMO_POLO_SEMANA "
      f"({len(resumo_polo_semana):,} linhas)")
print(f"\n=== TEMPO TOTAL DO PIPELINE: {time.time()-t_inicio:.1f}s ===")


# =============================================================================
# 8. CURVA DE SAVING POR MÓDULO — única aba adicional (não altera nada acima).
#
# Gráfico 1 — CH DEMANDADA por tamanho de grupo, aberto por módulo: quanto de
# CH cada tamanho exato de grupo (1, 2, 3... até 10+ alunos) demanda, uma
# série por módulo.
#
# Gráfico 2 — SAVING por cenário de corte, aberto por módulo: para cada
# limite N, o cenário é "só ofertar prática para grupos com N alunos OU MAIS"
# — o saving é a CH de tudo que ficaria de fora (grupos com MENOS de N
# alunos). Ou seja, LIMITE_ALUNOS_OFERTA = N representa o piso mínimo
# mantido, não um teto de corte.
# =============================================================================
t0 = time.time()

from openpyxl import load_workbook
from openpyxl.chart import LineChart, BarChart, Reference
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

def _escrever_dataframe_manual(ws, df, linha_inicial, coluna_inicial):
    """Escreve um DataFrame numa planilha openpyxl já aberta, sem passar
    pelo pandas ExcelWriter — evita reabrir/regravar o arquivo inteiro."""
    for j, col_nome in enumerate(df.columns):
        ws.cell(row=linha_inicial, column=coluna_inicial + j, value=col_nome)
    for i, row in enumerate(df.itertuples(index=False)):
        for j, valor in enumerate(row):
            ws.cell(row=linha_inicial + 1 + i, column=coluna_inicial + j, value=valor)

# Grão: 1 linha por oferta (grupo físico já formado, considerando sinergia).
ofertas_linha = detalhe[detalhe['LINHA_CONTABILIZADORA_OFERTA']][
    ['ID_OFERTA', 'MODULO', 'QTDE_ALUNOS_OFERTA', 'DEMANDA_CH_MODULO_LINHA']
].copy()

def _faixa_alunos_oferta(n):
    return '10+' if n >= 10 else str(int(n))

ofertas_linha['FAIXA_ALUNOS'] = ofertas_linha['QTDE_ALUNOS_OFERTA'].apply(_faixa_alunos_oferta)

_ordem_faixa = {str(i): i for i in range(1, 10)}
_ordem_faixa['10+'] = 10
_modulos_ordenados = sorted(ofertas_linha['MODULO'].unique())

# ---- Tabela do Gráfico 1: CH demandada por tamanho de grupo x módulo ----
_demanda_por_modulo = (
    ofertas_linha.groupby(['FAIXA_ALUNOS', 'MODULO'])['DEMANDA_CH_MODULO_LINHA']
    .sum().reset_index()
)
pivot_demanda_modulo = _demanda_por_modulo.pivot(
    index='FAIXA_ALUNOS', columns='MODULO', values='DEMANDA_CH_MODULO_LINHA'
).fillna(0)
pivot_demanda_modulo = pivot_demanda_modulo.reindex(
    sorted(pivot_demanda_modulo.index, key=lambda x: _ordem_faixa[x])
)
pivot_demanda_modulo = pivot_demanda_modulo[_modulos_ordenados].reset_index().rename(
    columns={'FAIXA_ALUNOS': 'QTDE_ALUNOS_NA_OFERTA'}
)
pivot_demanda_modulo.columns = [
    c if c == 'QTDE_ALUNOS_NA_OFERTA' else f'MODULO_{c}' for c in pivot_demanda_modulo.columns
]

# ---- Tabela do Gráfico 2: saving por cenário "manter >= N alunos", x módulo ----
# CH_ECONOMIZADA(N) = soma da CH de todas as ofertas com QTDE_ALUNOS_OFERTA < N
# (são as que deixariam de ser ofertadas nesse cenário).
_limites_piso = list(range(1, 11))  # N = 1 a 10 ("manter >= N alunos")
_linhas_saving_mod = []
for _modulo in _modulos_ordenados:
    _sub = ofertas_linha[ofertas_linha['MODULO'] == _modulo]
    for _n in _limites_piso:
        _filtro_cortado = _sub['QTDE_ALUNOS_OFERTA'] < _n
        _linhas_saving_mod.append({
            'MODULO': _modulo,
            'LIMITE_ALUNOS_OFERTA': _n,
            'CH_ECONOMIZADA': _sub.loc[_filtro_cortado, 'DEMANDA_CH_MODULO_LINHA'].sum(),
        })
_saving_mod_df = pd.DataFrame(_linhas_saving_mod)
pivot_saving_modulo = _saving_mod_df.pivot(
    index='LIMITE_ALUNOS_OFERTA', columns='MODULO', values='CH_ECONOMIZADA'
)
pivot_saving_modulo = pivot_saving_modulo[_modulos_ordenados].reset_index()
pivot_saving_modulo.columns = [
    c if c == 'LIMITE_ALUNOS_OFERTA' else f'MODULO_{c}' for c in pivot_saving_modulo.columns
]
pivot_saving_modulo = pivot_saving_modulo.rename(
    columns={'LIMITE_ALUNOS_OFERTA': 'SO_OFERTAR_PARA_N_ALUNOS_OU_MAIS'}
)
# Coluna TOTAL: soma de TODOS os módulos presentes na base (3 em 2026, não 4) — vira a curva agregada
# de economia, sobreposta às barras por módulo no Gráfico 2.
pivot_saving_modulo['TOTAL'] = pivot_saving_modulo[
    [f'MODULO_{m}' for m in _modulos_ordenados]
].sum(axis=1)

# ---- Escreve a aba única e os 2 gráficos (abertura única do workbook) ----
NOME_ABA_MODULO = 'CURVA_SAVING_POR_MODULO'
_wb2 = load_workbook(CAMINHO_SAIDA)
_ws2 = _wb2.create_sheet(NOME_ABA_MODULO)

_escrever_dataframe_manual(_ws2, pivot_demanda_modulo, linha_inicial=3, coluna_inicial=1)
_col2_start = len(pivot_demanda_modulo.columns) + 2
_escrever_dataframe_manual(_ws2, pivot_saving_modulo, linha_inicial=3, coluna_inicial=_col2_start)

# O rótulo diz "tamanho exato": a tabela NÃO é acumulada — a coluna do 3 são
# os grupos com exatamente 3 alunos, não os com até 3.
_ws2['A1'] = 'CH DEMANDADA por tamanho EXATO de grupo, aberto por módulo (não acumulado)'
_ws2['A1'].font = Font(name='Calibri', size=12, bold=True, color='1F4E78')
_ws2.cell(row=1, column=_col2_start,
          value='SAVING se só ofertar para grupos com N alunos ou mais, aberto por módulo')
_ws2.cell(row=1, column=_col2_start).font = Font(name='Calibri', size=12, bold=True, color='1F4E78')

_n_col1 = len(pivot_demanda_modulo.columns)
_n_col2 = len(pivot_saving_modulo.columns)
for _col in list(range(1, _n_col1 + 1)) + list(range(_col2_start, _col2_start + _n_col2)):
    _ws2.cell(row=3, column=_col).font = Font(bold=True)
    _ws2.column_dimensions[get_column_letter(_col)].width = 18

_linhas_tabela1 = len(pivot_demanda_modulo)  # 10 (faixas 1..9, 10+)
_linhas_tabela2 = len(pivot_saving_modulo)   # 10 (N de 1 a 10)

_grafico_demanda_mod = BarChart()
_grafico_demanda_mod.type = "col"
_grafico_demanda_mod.grouping = "clustered"
_grafico_demanda_mod.title = "CH demandada por tamanho de grupo, por módulo"
_grafico_demanda_mod.y_axis.title = "Horas de demanda"
_grafico_demanda_mod.x_axis.title = "Alunos na oferta"
_grafico_demanda_mod.style = 10
_dados_g1 = Reference(_ws2, min_col=2, max_col=_n_col1, min_row=3, max_row=3 + _linhas_tabela1)
_cats_g1 = Reference(_ws2, min_col=1, min_row=4, max_row=3 + _linhas_tabela1)
_grafico_demanda_mod.add_data(_dados_g1, titles_from_data=True)
_grafico_demanda_mod.set_categories(_cats_g1)
_grafico_demanda_mod.height, _grafico_demanda_mod.width = 10, 20
_ws2.add_chart(_grafico_demanda_mod, "A16")

_grafico_saving_mod = BarChart()
_grafico_saving_mod.type = "col"
_grafico_saving_mod.grouping = "clustered"
_grafico_saving_mod.title = "Saving por cenário 'só ofertar para N alunos ou mais', por módulo"
_grafico_saving_mod.y_axis.title = "Horas economizadas"
_grafico_saving_mod.x_axis.title = "N (mínimo de alunos mantido na oferta)"
_grafico_saving_mod.style = 12
# Barras: só as colunas de módulo (a última coluna, TOTAL, fica de fora daqui
# e vira a linha combinada logo abaixo).
_col_dados_g2_min = _col2_start + 1
_col_dados_g2_max = _col2_start + _n_col2 - 2  # exclui a coluna TOTAL (última)
_dados_g2 = Reference(_ws2, min_col=_col_dados_g2_min, max_col=_col_dados_g2_max,
                       min_row=3, max_row=3 + _linhas_tabela2)
_cats_g2 = Reference(_ws2, min_col=_col2_start, min_row=4, max_row=3 + _linhas_tabela2)
_grafico_saving_mod.add_data(_dados_g2, titles_from_data=True)
_grafico_saving_mod.set_categories(_cats_g2)
_grafico_saving_mod.height, _grafico_saving_mod.width = 10, 20

# Linha: coluna TOTAL (soma dos módulos presentes), combinada no mesmo gráfico.
_grafico_total_linha = LineChart()
_col_total = _col2_start + _n_col2 - 1  # última coluna da tabela (TOTAL)
_dados_total = Reference(_ws2, min_col=_col_total, min_row=3, max_row=3 + _linhas_tabela2)
_grafico_total_linha.add_data(_dados_total, titles_from_data=True)
_grafico_total_linha.set_categories(_cats_g2)
_serie_total = _grafico_total_linha.series[0]
_serie_total.smooth = False
_serie_total.marker.symbol = "circle"
_serie_total.graphicalProperties.line.width = 25000  # linha mais grossa (EMU)

_grafico_saving_mod += _grafico_total_linha
_ws2.add_chart(_grafico_saving_mod, get_column_letter(_col2_start) + "16")

_wb2.save(CAMINHO_SAIDA)

print(f"[Curva por módulo] Aba '{NOME_ABA_MODULO}' criada com 2 gráficos "
      f"— {time.time()-t0:.1f}s")

## VALIDAÇÃO FINAL — conferências automáticas

Rodam sobre o resultado e falham alto se alguma premissa do modelo tiver sido quebrada. Vale rodar sempre que a base de entrada for atualizada.

In [ ]:
print("=" * 78)
print(f"VALIDAÇÃO DO RESULTADO — MODO_ANALISE = '{MODO_ANALISE}'")
print("=" * 78)

por_modulo = resultado.groupby('MODULO').agg(
    ALUNOS=('QTDE_ALUNOS_MODULO_LINHA', 'sum'),
    GRUPOS=('QTDE_GRUPOS_FISICOS_MODULO_LINHA', 'sum'),
    DEMANDA_H=('DEMANDA_CH_MODULO_LINHA', 'sum'),
    CAPACIDADE_H=('CH_CAPACIDADE_MODULO_RATEADA_LINHA', 'sum'),
)
por_modulo['DELTA_H'] = por_modulo['CAPACIDADE_H'] - por_modulo['DEMANDA_H']
por_modulo = por_modulo.round(0)
por_modulo['COBERTURA'] = (por_modulo['CAPACIDADE_H'] / por_modulo['DEMANDA_H']).round(3)
print("\n1. Demanda x capacidade por módulo:")
print(por_modulo.to_string())

# O delta agregado ESCONDE o problema real: sobra numa cidade não cobre falta
# em outra. O número acionável é a soma apenas dos déficits.
deficit = resumo_modulo_curso[resumo_modulo_curso['DELTA_CH_MODULO_CURSO'] < 0]
gap = deficit.groupby('MODULO').agg(
    PARES_CIDADE_CURSO_EM_DEFICIT=('DELTA_CH_MODULO_CURSO', 'size'),
    HORAS_FALTANTES=('DELTA_CH_MODULO_CURSO', lambda s: -s.sum()),
)
gap['TUTORES_EQUIVALENTES_20H'] = (gap['HORAS_FALTANTES'] / (20 * SEMANAS_DO_MODULO)).round(1)
print("\n2. Déficit real (soma apenas dos negativos — sobra de uma cidade não "
      "cobre falta de outra):")
print(gap.round(0).to_string())

excesso = resumo_tutor_modulo[
    resumo_tutor_modulo['CH_NAO_UTILIZADA_TOTAL']
    > resumo_tutor_modulo['CH_CAPACIDADE_TOTAL_MODULO_TUTOR'] + 0.01
]
print(f"\n3. Tutores com ociosidade maior que a própria capacidade: {len(excesso)} "
      f"(esperado: 0)")
assert len(excesso) == 0, "Rateio de capacidade inconsistente!"

print(f"\n4. Capacidade distribuída em polos / capacidade da cidade: "
      f"{cap_polo/max(cap_cidade,1):.2%} (esperado: 100% — o pool é municipal e "
      f"é integralmente distribuído)")
assert abs(cap_polo - cap_cidade) < max(cap_cidade * 1e-6, 1), \
    "A soma dos polos não reconcilia com a cidade!"

dem_cidade = resumo_modulo_curso['DEMANDA_CH_MODULO_CURSO'].sum()
dem_polo = resumo_polo_modulo['DEMANDA_CH_MODULO_CURSO_POLO'].sum()
print(f"\n4B. Demanda somada nos polos / demanda da cidade: "
      f"{dem_polo/max(dem_cidade,1):.2%} (esperado: 100%)")
assert abs(dem_polo - dem_cidade) < max(dem_cidade * 1e-6, 1), \
    "A demanda dos polos não reconcilia com a cidade!"

# A CH de mentoria (preparação) está CONTIDA na CH do tutor, não é adicional.
# Se algum curso mostrar mentoria maior que a demanda, a base de PRATICAS tem
# CH_SEMANAL_PRATICA_DO_TUTOR menor que a duração da prática.
ment_maior = resumo_polo_modulo[
    resumo_polo_modulo['CH_MENTORIA_MODULO_CURSO_POLO']
    > resumo_polo_modulo['DEMANDA_CH_MODULO_CURSO_POLO'] + 0.01
]
print(f"\n4C. Linhas com CH de mentoria maior que a demanda: {len(ment_maior)} "
      f"(esperado: 0 — a mentoria está CONTIDA na CH do tutor, não é um "
      f"adicional; nunca some demanda + mentoria)")
assert len(ment_maior) == 0, "CH de preparação excede a CH total do tutor!"

print("\n5. Status por curso no módulo (cidade):")
print(resumo_modulo_curso.groupby('STATUS_MODULO_CURSO').size().to_string())

print("\n6. Status por curso no polo (aba RESUMO_POLO_MODULO):")
print(resumo_polo_modulo.groupby('STATUS_MODULO_CURSO_POLO').size().to_string())

print("\n7. Reconciliação módulo x semana (Etapa 9) — soma_semanal x 10 = soma_modular:")
_pares = [
    ('ANALISE_DISCIPLINA_SEMANA', resultado, analise_disciplina_semana,
     HORA_DISCIPLINA_MODULO),
    ('RESUMO_POLO_SEMANA', resumo_polo_modulo, resumo_polo_semana, HORA_POLO_MODULO),
]
for _nome, _mod, _sem, _cols in _pares:
    for _c in _cols:
        if _c not in _mod.columns:
            continue
        _c_sem = (_c.replace('_MODULO', '_SEMANA_MEDIA') if '_MODULO' in _c
                  else f'{_c}_SEMANA_MEDIA')
        _tot_mod = _mod[_c].sum()
        _tot_sem = _sem[_c_sem].sum() * SEMANAS_DO_MODULO
        _ok = abs(_tot_mod - _tot_sem) < max(abs(_tot_mod) * 1e-9, 1e-6)
        print(f"   {'OK ' if _ok else 'ERRO'} {_nome:26s} {_c:45s} "
              f"{_tot_mod:>14,.2f} = {_tot_sem:>14,.2f}")
        assert _ok, f"Proporção semanal quebrada em {_nome}.{_c}!"

print("\n" + "=" * 78)
print("Validação concluída.")
print("=" * 78)
